# Bansali SmartNest - SEE. MEASURE. OPTIMIZE. CUT.
**Camera -> measured material map -> nested parts -> validated cutting DXF**, in the operator's five clicks:

| Click | Step | What the software does |
|---|---|---|
| 1 | **SCAN BED** | capture the bed, map pixels to machine millimetres (homography + lens model) |
| 2 | **CONFIRM MATERIAL** | detect the sheet, every existing cut-out and edge notch; show a quality score; allow corrections |
| 3 | **ADD JOB** | read the part DXF, reject open / self-intersecting contours, count quantities |
| 4 | **OPTIMIZE** | nest the parts into the *remaining* material (No-Fit-Polygon kernel + evolutionary search) |
| 5 | **EXPORT** | write the DXF in machine mm, read it back, compare, and only then release it |

`CAPTURE_MODE = 'demo'` uses synthetic beds whose true geometry is known, so the notebook reports measured errors
instead of "looks right". Measured on those scenes: sheet area within about 0.05 %, circle diameters within 0.6 mm,
median edge error 0.1 mm at 1.8 mm/px. **These numbers are from simulation - validate on real Bansali machine
images before trusting the vision stage in production.**

**Where to run:** Google Colab (colab.research.google.com -> File -> Upload notebook -> Runtime -> Run all) supports
everything, including the live webcam and click-the-corners widgets. VS Code / local Jupyter (pick a Python 3.10+
kernel) runs the full pipeline in `demo` mode and with your own images; the webcam and click widgets are Colab-only.

In [ ]:
#@title Step 0 - install (about 30 s)
%pip install -q "shapely>=2.1" ezdxf opencv-python matplotlib

In [ ]:
%%writefile smartnest_vision.py
"""
Bansali SmartNest - vision core (optimized rewrite of the Colab prototype).

Every measurement ends in *bed millimetres*. The pipeline:

    raw camera frame
      -> lens undistortion           camera_matrix + dist_coeffs (chessboard, once per camera)
      -> homography  image -> mm     4+ reference points (clicks or ArUco markers), once per mount
      -> orthographic resample       ONE remap at the camera's native resolution (cached maps)
      -> material segmentation       empty-bed reference difference, or Otsu on luminance
      -> contour hierarchy           outer boundary = sheet, child contours = existing cut-outs
      -> Shapely geometry (mm)       material = sheet - union(cut-outs), half-pixel corrected
      -> classification, confidence
      -> DXF (Y-up, mm) + validation by reading the file back

Coordinate frames (see README):
    image px : OpenCV convention, pixel (col j, row i) has its centre at (x=j, y=i)
    bed mm   : origin at calibration corner #1 (image top-left), x -> corner #2, y -> corner #4
    ortho px : pixel (u, v) has its centre at bed mm ((u + 0.5) * s, (v + 0.5) * s)
    CAD mm   : right-handed, Y up; X = x, Y = bed_h - y   (origin = bottom-left of the bed)

Nothing here assumes a global pixel/mm ratio, a sheet that fills the bed, or round cut-outs.
"""
from __future__ import annotations

import json
import math
from dataclasses import dataclass, field
from typing import Any, Sequence

import cv2
import numpy as np
import shapely
from shapely.geometry import LinearRing, MultiPolygon, Point, Polygon, box
from shapely.geometry.polygon import orient
from shapely.ops import unary_union
from shapely.validation import make_valid

CORNER_NAMES = ("top-left", "top-right", "bottom-right", "bottom-left")


class VisionError(RuntimeError):
    """Raised with an operator-readable message when a scan cannot be trusted."""


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
@dataclass
class VisionConfig:
    mm_per_px: float | None = None     # ortho resolution; None = match the camera's native resolution
    segmentation: str = "auto"         # "auto" (reference if given, else intensity) | "reference" | "intensity"
    polarity: str = "bright"           # intensity mode: sheet "bright"er or "dark"er than the bed
    min_feature_mm: float = 6.0        # thinner structures are removed: slat tops, scratches, dust
    min_sheet_area_mm2: float = 50_000.0
    max_hole_fraction: float = 0.5     # a "sheet" whose biggest hole is larger than this is the bed around a sheet
    ambiguity_ratio: float = 0.15      # a 2nd blob this large (vs the sheet) is reported as ambiguous
    reference_min_delta: float = 18.0  # min Lab colour difference that counts as "something is on the bed"
    simplify_mm: float = 0.02          # drops collinear points only: a coarser Douglas-Peucker pass
                                       # keeps rounded corner points and shifts whole edges
    circle_rms_mm: float = 0.6         # max RMS radial residual to call a cut-out a circle ...
    circle_rms_frac: float = 0.015     # ... or this fraction of the radius, whichever is larger
    rect_fill: float = 0.97            # area / minimum-rotated-rectangle area to call it a rectangle
    edge_notch_min_area_mm2: float = 400.0
    confirm_below: float = 0.85        # quality score below this requires operator confirmation


# ---------------------------------------------------------------------------
# Small numeric helpers
# ---------------------------------------------------------------------------
def _pts(a) -> np.ndarray:
    return np.asarray(a, dtype=np.float64).reshape(-1, 2)


def apply_homography(H: np.ndarray, pts) -> np.ndarray:
    p = _pts(pts)
    q = np.c_[p, np.ones(len(p))] @ np.asarray(H, dtype=np.float64).T
    return q[:, :2] / q[:, 2:3]


def _shoelace(p: np.ndarray) -> float:
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def distort_pixels(pts, K, D) -> np.ndarray:
    """Ideal (pinhole) pixel coordinates -> where the real, distorting lens puts them."""
    p = _pts(pts)
    K = np.asarray(K, dtype=np.float64)
    xn = (p[:, 0] - K[0, 2]) / K[0, 0]
    yn = (p[:, 1] - K[1, 2]) / K[1, 1]
    obj = np.c_[xn, yn, np.ones(len(p))].reshape(-1, 1, 3)
    img, _ = cv2.projectPoints(obj, np.zeros(3), np.zeros(3), K, np.asarray(D, dtype=np.float64))
    return img.reshape(-1, 2)


def undistort_pixels(pts, K, D, iterations: int = 6) -> np.ndarray:
    """Distorted pixels -> ideal pixels. cv2.undistortPoints plus fixed-point refinement,
    because its default 5 iterations are not enough near the corners of wide-angle webcams."""
    p = _pts(pts)
    K = np.asarray(K, dtype=np.float64)
    D = np.asarray(D, dtype=np.float64)
    u = cv2.undistortPoints(p.reshape(-1, 1, 2), K, D, P=K).reshape(-1, 2)
    for _ in range(iterations):
        u = u + (p - distort_pixels(u, K, D))
    return u


def _otsu(values: np.ndarray) -> tuple[int, float]:
    """Otsu threshold on uint8 values and its separability eta = between-class / total variance
    (1.0 = two perfectly separated populations; ~0 = no structure)."""
    hist = np.bincount(values.ravel(), minlength=256).astype(np.float64)
    p = hist / max(hist.sum(), 1.0)
    levels = np.arange(256, dtype=np.float64)
    omega = np.cumsum(p)
    mu = np.cumsum(p * levels)
    mu_t = mu[-1]
    var_t = float(np.sum(p * (levels - mu_t) ** 2))
    with np.errstate(divide="ignore", invalid="ignore"):
        sigma_b = (mu_t * omega - mu) ** 2 / (omega * (1.0 - omega))
    sigma_b = np.nan_to_num(sigma_b)
    t = int(np.argmax(sigma_b))
    return t, float(sigma_b[t] / var_t) if var_t > 0 else 0.0


def fit_circle(pts) -> tuple[float, float, float, float]:
    """Algebraic least-squares circle (Kasa). Returns cx, cy, r, rms radial residual."""
    p = _pts(pts)
    m = p.mean(axis=0)
    q = p - m
    A = np.c_[2.0 * q, np.ones(len(q))]
    b = (q ** 2).sum(axis=1)
    (a, bb, c), *_ = np.linalg.lstsq(A, b, rcond=None)
    r = math.sqrt(max(c + a * a + bb * bb, 0.0))
    resid = np.hypot(q[:, 0] - a, q[:, 1] - bb) - r
    return float(m[0] + a), float(m[1] + bb), r, float(np.sqrt(np.mean(resid ** 2)))


def _ring_samples(ring, step: float) -> np.ndarray:
    ring = LinearRing(ring) if not isinstance(ring, LinearRing) else ring
    n = int(np.clip(ring.length / max(step, 1e-6), 32, 20000))
    d = np.linspace(0.0, ring.length, n, endpoint=False)
    return shapely.get_coordinates(shapely.line_interpolate_point(ring, d))


def _polygons(geom) -> list[Polygon]:
    """All polygonal parts of any geometry (make_valid may return collections)."""
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, Polygon):
        return [geom]
    if isinstance(geom, MultiPolygon):
        return list(geom.geoms)
    if hasattr(geom, "geoms"):
        return [p for g in geom.geoms for p in _polygons(g)]
    return []


def _as_area(geom):
    parts = _polygons(geom)
    if not parts:
        return Polygon()
    return parts[0] if len(parts) == 1 else MultiPolygon(parts)


# ---------------------------------------------------------------------------
# Calibration
# ---------------------------------------------------------------------------
@dataclass
class BedCalibration:
    bed_w_mm: float
    bed_h_mm: float
    H: np.ndarray                      # 3x3: *undistorted* image px -> bed mm
    image_size: tuple[int, int]        # (width, height) of camera frames
    camera_matrix: np.ndarray | None = None
    dist_coeffs: np.ndarray | None = None
    rms_mm: float | None = None        # residual at the reference points; None = exactly 4 points
    n_points: int = 4
    source: str = "manual"
    warnings: list[str] = field(default_factory=list)

    @property
    def has_lens_model(self) -> bool:
        return self.camera_matrix is not None and self.dist_coeffs is not None

    def image_to_bed(self, pts_px) -> np.ndarray:
        p = _pts(pts_px)
        if self.has_lens_model:
            p = undistort_pixels(p, self.camera_matrix, self.dist_coeffs)
        return apply_homography(self.H, p)

    def bed_to_image(self, pts_mm) -> np.ndarray:
        p = apply_homography(np.linalg.inv(self.H), pts_mm)
        if self.has_lens_model:
            p = distort_pixels(p, self.camera_matrix, self.dist_coeffs)
        return p

    def bed_corners_mm(self) -> np.ndarray:
        w, h = self.bed_w_mm, self.bed_h_mm
        return np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float64)

    def native_mm_per_px(self) -> float:
        """Average size of one camera pixel on the bed plane."""
        px = apply_homography(np.linalg.inv(self.H), self.bed_corners_mm())
        return math.sqrt(self.bed_w_mm * self.bed_h_mm / _shoelace(px))

    def to_dict(self) -> dict[str, Any]:
        return {
            "bed_w_mm": self.bed_w_mm, "bed_h_mm": self.bed_h_mm,
            "homography_img_to_mm": np.asarray(self.H).tolist(),
            "image_size": list(self.image_size),
            "camera_matrix": None if self.camera_matrix is None else np.asarray(self.camera_matrix).tolist(),
            "dist_coeffs": None if self.dist_coeffs is None else np.asarray(self.dist_coeffs).ravel().tolist(),
            "rms_mm": self.rms_mm, "n_points": self.n_points, "source": self.source,
            "bed_corners_px": self.bed_to_image(self.bed_corners_mm()).round(2).tolist(),
            "native_mm_per_px": round(self.native_mm_per_px(), 4),
        }

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "BedCalibration":
        return cls(
            bed_w_mm=float(d["bed_w_mm"]), bed_h_mm=float(d["bed_h_mm"]),
            H=np.array(d["homography_img_to_mm"], dtype=np.float64),
            image_size=tuple(d["image_size"]),
            camera_matrix=None if d.get("camera_matrix") is None else np.array(d["camera_matrix"], dtype=np.float64),
            dist_coeffs=None if d.get("dist_coeffs") is None else np.array(d["dist_coeffs"], dtype=np.float64),
            rms_mm=d.get("rms_mm"), n_points=int(d.get("n_points", 4)), source=d.get("source", "file"),
        )

    def save(self, path: str) -> None:
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def load(cls, path: str) -> "BedCalibration":
        with open(path) as f:
            return cls.from_dict(json.load(f))


def order_corners_clockwise(pts, keep_first: bool = True) -> tuple[np.ndarray, bool]:
    """Put 4 clicked corners in clockwise image order (TL, TR, BR, BL for an upright view).

    A crossed or mirrored click order would produce a twisted or mirrored homography - and a
    mirrored cutting file. If keep_first, the first click stays first (it defines the origin).
    Returns (ordered, changed)."""
    p = _pts(pts)
    if len(p) != 4:
        raise VisionError("Exactly 4 bed corners are required.")
    c = p.mean(axis=0)
    ang = np.arctan2(p[:, 1] - c[1], p[:, 0] - c[0])   # image y is down -> ascending angle = clockwise
    order = list(np.argsort(ang))
    start = order.index(0) if keep_first else order.index(int(np.argmin(p.sum(axis=1))))
    order = order[start:] + order[:start]
    q = p[order]
    if not cv2.isContourConvex(q.astype(np.float32).reshape(-1, 1, 2)):
        raise VisionError("The 4 bed corners do not form a convex shape - re-click the corners.")
    return q, order != [0, 1, 2, 3]


def refine_corners(gray: np.ndarray, pts, window_px: int = 5, max_shift_px: float = 2.0):
    """Sub-pixel snap with a safety limit. cornerSubPix is designed for chessboard saddle points;
    on an L-shaped bed corner next to slats it can slide along an edge, so any point that moves
    more than max_shift_px keeps the operator's click. Returns (points, shifts_px, accepted)."""
    p = _pts(pts).astype(np.float32).reshape(-1, 1, 2)
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.01)
    refined = cv2.cornerSubPix(gray, p.copy(), (window_px, window_px), (-1, -1), crit).reshape(-1, 2)
    orig = p.reshape(-1, 2)
    shift = np.hypot(*(refined - orig).T)
    ok = shift <= max_shift_px
    out = np.where(ok[:, None], refined, orig).astype(np.float64)
    return out, shift, ok


def calibrate_bed(image_pts, bed_w_mm: float, bed_h_mm: float, image_size: tuple[int, int],
                  bed_pts_mm=None, camera_matrix=None, dist_coeffs=None,
                  source: str = "manual") -> BedCalibration:
    """Homography from N >= 4 correspondences (image px -> bed mm).

    With exactly 4 points the homography fits them perfectly, so its error is *unknown*.
    With more points (e.g. mid-edge marks or ArUco corners) the residual is a real accuracy check."""
    img = _pts(image_pts)
    warnings: list[str] = []
    if bed_pts_mm is None:
        img, changed = order_corners_clockwise(img)
        if changed:
            warnings.append("Corner click order was corrected to clockwise (TL, TR, BR, BL).")
        mm = np.array([[0, 0], [bed_w_mm, 0], [bed_w_mm, bed_h_mm], [0, bed_h_mm]], dtype=np.float64)
    else:
        mm = _pts(bed_pts_mm)
    if len(img) != len(mm) or len(img) < 4:
        raise VisionError("Calibration needs at least 4 matching image/bed points.")
    und = undistort_pixels(img, camera_matrix, dist_coeffs) if camera_matrix is not None else img
    if len(img) == 4:
        H = cv2.getPerspectiveTransform(und.astype(np.float32), mm.astype(np.float32)).astype(np.float64)
        rms = None
        warnings.append("Only 4 reference points: calibration error cannot be measured. "
                        "Add fiducials (ArUco markers or marked points) for a verified calibration.")
    else:
        H, _ = cv2.findHomography(und, mm, 0)
        if H is None:
            raise VisionError("Calibration points are degenerate (collinear or duplicated).")
        rms = float(np.sqrt(np.mean(np.sum((apply_homography(H, und) - mm) ** 2, axis=1))))
    back = apply_homography(np.linalg.inv(H), [[0, 0], [bed_w_mm, 0], [bed_w_mm, bed_h_mm], [0, bed_h_mm]])
    if not cv2.isContourConvex(back.astype(np.float32).reshape(-1, 1, 2)):
        raise VisionError("Calibration is folded or mirrored - check the reference points.")
    if rms is not None and rms > 1.5:
        warnings.append(f"Calibration residual is {rms:.2f} mm - check lens calibration or reference points.")
    return BedCalibration(bed_w_mm, bed_h_mm, H, tuple(image_size),
                          None if camera_matrix is None else np.asarray(camera_matrix, dtype=np.float64),
                          None if dist_coeffs is None else np.asarray(dist_coeffs, dtype=np.float64),
                          rms, len(img), source, warnings)


def aruco_layout(centers_mm: dict[int, tuple[float, float]], size_mm: float) -> dict[int, np.ndarray]:
    """Bed-mm corners (TL, TR, BR, BL - ArUco order) of square markers mounted axis-aligned."""
    h = size_mm / 2.0
    return {i: np.array([[x - h, y - h], [x + h, y - h], [x + h, y + h], [x - h, y + h]])
            for i, (x, y) in centers_mm.items()}


def calibrate_from_aruco(image: np.ndarray, layout_mm: dict[int, np.ndarray], bed_w_mm: float,
                         bed_h_mm: float, dictionary: int = cv2.aruco.DICT_4X4_50,
                         camera_matrix=None, dist_coeffs=None) -> BedCalibration:
    """Automatic, self-checking bed calibration from ArUco markers fixed to the machine frame.
    Re-run it on every scan and a bumped camera is detected instead of silently mis-measuring."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    det = cv2.aruco.ArucoDetector(cv2.aruco.getPredefinedDictionary(dictionary),
                                  cv2.aruco.DetectorParameters())
    corners, ids, _ = det.detectMarkers(gray)
    img_pts, mm_pts = [], []
    if ids is not None:
        crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.01)
        for c, i in zip(corners, ids.ravel()):
            if int(i) in layout_mm:
                c = cv2.cornerSubPix(gray, c.reshape(-1, 1, 2).astype(np.float32), (5, 5), (-1, -1), crit)
                img_pts.append(c.reshape(4, 2))
                mm_pts.append(layout_mm[int(i)])
    if len(img_pts) < 2:
        raise VisionError(f"Only {len(img_pts)} calibration marker(s) visible - at least 2 are required.")
    cal = calibrate_bed(np.vstack(img_pts), bed_w_mm, bed_h_mm, gray.shape[::-1], np.vstack(mm_pts),
                        camera_matrix, dist_coeffs, source=f"aruco x{len(img_pts)}")
    return cal


def calibrate_lens_chessboard(images: Sequence[np.ndarray], pattern_size=(9, 6), square_mm: float = 25.0):
    """Standard OpenCV intrinsic calibration from >= 8 chessboard photos (varied tilt/position).
    Returns camera_matrix, dist_coeffs, rms_px. Run once per camera + lens + focus setting."""
    objp = np.zeros((pattern_size[0] * pattern_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1, 2) * square_mm
    obj_pts, img_pts, size = [], [], None
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.001)
    for im in images:
        gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if im.ndim == 3 else im
        size = gray.shape[::-1]
        ok, c = cv2.findChessboardCorners(gray, pattern_size, None)
        if ok:
            obj_pts.append(objp)
            img_pts.append(cv2.cornerSubPix(gray, c, (11, 11), (-1, -1), crit))
    if len(obj_pts) < 5:
        raise VisionError(f"Chessboard found in only {len(obj_pts)} photo(s); need at least 5 (8+ recommended).")
    rms, K, D, _, _ = cv2.calibrateCamera(obj_pts, img_pts, size, None, None)
    return K, D.ravel(), float(rms)


# ---------------------------------------------------------------------------
# Orthographic rectification
# ---------------------------------------------------------------------------
class Rectifier:
    """Resamples camera frames into a metric top-down bed image.

    One interpolation from the raw frame (undistortion + homography folded into a single cached
    map), so live video costs one cv2.remap per frame. Default resolution = the camera's own
    resolution on the bed, so no detail is thrown away and no fake detail is invented."""

    def __init__(self, calib: BedCalibration, mm_per_px: float | None = None):
        s = mm_per_px or float(np.clip(round(calib.native_mm_per_px() / 0.05) * 0.05, 0.25, 4.0))
        self.calib = calib
        self.s = float(s)
        self.w = int(math.ceil(calib.bed_w_mm / self.s - 1e-9))
        self.h = int(math.ceil(calib.bed_h_mm / self.s - 1e-9))
        A = np.array([[1 / self.s, 0, -0.5], [0, 1 / self.s, -0.5], [0, 0, 1]], dtype=np.float64)
        self.M = A @ calib.H                       # undistorted image px -> ortho px
        self._maps = None
        if calib.has_lens_model:
            u, v = np.meshgrid(np.arange(self.w), np.arange(self.h))
            mm = np.c_[(u.ravel() + 0.5) * self.s, (v.ravel() + 0.5) * self.s]
            src = calib.bed_to_image(mm).astype(np.float32)
            self._maps = cv2.convertMaps(src[:, 0].reshape(self.h, self.w), src[:, 1].reshape(self.h, self.w),
                                         cv2.CV_16SC2)
        W, H = calib.image_size
        self.valid = self.warp(np.full((H, W), 255, np.uint8), cv2.INTER_NEAREST) > 0

    def warp(self, image: np.ndarray, interpolation: int = cv2.INTER_LINEAR) -> np.ndarray:
        if image is None or image.size == 0:
            raise VisionError("No camera image.")
        if (image.shape[1], image.shape[0]) != tuple(self.calib.image_size):
            raise VisionError(f"Image is {image.shape[1]}x{image.shape[0]} px but the calibration was made "
                              f"for {self.calib.image_size[0]}x{self.calib.image_size[1]} px.")
        if self._maps is not None:
            return cv2.remap(image, self._maps[0], self._maps[1], interpolation, borderMode=cv2.BORDER_CONSTANT)
        return cv2.warpPerspective(image, self.M, (self.w, self.h), flags=interpolation,
                                   borderMode=cv2.BORDER_CONSTANT)

    def px_to_mm(self, pts) -> np.ndarray:
        return (_pts(pts) + 0.5) * self.s

    def mm_to_px(self, pts) -> np.ndarray:
        return _pts(pts) / self.s - 0.5

    @property
    def bed_box(self) -> Polygon:
        return box(0.0, 0.0, self.calib.bed_w_mm, self.calib.bed_h_mm)


# ---------------------------------------------------------------------------
# Segmentation: which ortho pixels are material?
# ---------------------------------------------------------------------------
def segment_material(ortho: np.ndarray, rect: Rectifier, cfg: VisionConfig,
                     reference_ortho: np.ndarray | None = None):
    """Returns (mask, feature, info). `feature` is oriented so material is HIGH; it is what the
    sub-pixel edge refinement later measures the 50 % crossing on."""
    mode = cfg.segmentation
    if mode == "auto":
        mode = "reference" if reference_ortho is not None else "intensity"
    blur = cv2.GaussianBlur(ortho, (0, 0), 0.5)
    if mode == "reference":
        if reference_ortho is None:
            raise VisionError("Reference segmentation needs an empty-bed image.")
        a = cv2.cvtColor(blur, cv2.COLOR_BGR2LAB).astype(np.float32)
        b = cv2.cvtColor(cv2.GaussianBlur(reference_ortho, (0, 0), 0.5), cv2.COLOR_BGR2LAB).astype(np.float32)
        feat = np.clip(np.linalg.norm(a - b, axis=2) * 2.0, 0, 255).astype(np.uint8)   # 2 units per dE
        t, eta = _otsu(feat[rect.valid])
        t = max(t, int(cfg.reference_min_delta * 2.0))
        mask = feat > t
        polarity = "changed"
    elif mode == "intensity":
        if cfg.polarity not in ("bright", "dark"):
            raise VisionError("polarity must be 'bright' or 'dark'.")
        feat = cv2.cvtColor(blur, cv2.COLOR_BGR2LAB)[:, :, 0]
        if cfg.polarity == "dark":
            feat = 255 - feat
        t, eta = _otsu(feat[rect.valid])
        mask = feat > t
        polarity = cfg.polarity
    else:
        raise VisionError(f"Unknown segmentation mode '{mode}'.")
    mask = (mask & rect.valid).astype(np.uint8) * 255
    k = max(3, int(round(cfg.min_feature_mm / rect.s)) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    # OPEN drops thin bright things (slat tops, glints); CLOSE heals thin dark things (scratches).
    # Reference mode closes first: wherever the empty-bed image ramps through the sheet's own
    # value (slat edges) the difference is ~0, leaving 1 px "unchanged" cracks inside the sheet.
    ops = (cv2.MORPH_CLOSE, cv2.MORPH_OPEN) if mode == "reference" else (cv2.MORPH_OPEN, cv2.MORPH_CLOSE)
    for op in ops:
        mask = cv2.morphologyEx(mask, op, kernel)
    return mask, feat, {"mode": mode, "polarity": polarity, "threshold": int(t), "separability": round(eta, 3),
                        "kernel_px": k}


def refine_ring(coords_mm, feat_f32: np.ndarray, rect: Rectifier, reach_px: float = 4.0,
                max_shift_px: float = 1.5, min_contrast: float = 20.0):
    """Sub-pixel edge refinement. At every ~1 px along the ring, sample the feature profile along
    the normal and move the point to where it crosses the midpoint between the LOCAL material
    level and the LOCAL bed level. A global threshold puts the edge in the wrong place wherever
    lighting falls off, glare brightens the sheet, or a bright slat sits next to the edge.

    The ring must be oriented so its left normal points into material (see shapely `orient`).
    Returns (refined coords in mm, per-sample 'clear step' flags, per-sample shifts in px)."""
    ring = LinearRing(coords_mm)
    p = _ring_samples(ring, rect.s)
    k = 2
    t = np.roll(p, -k, axis=0) - np.roll(p, k, axis=0)
    t /= np.maximum(np.linalg.norm(t, axis=1, keepdims=True), 1e-9)
    n_in = np.c_[-t[:, 1], t[:, 0]]
    offs = np.arange(-reach_px, reach_px + 1e-9, 0.5)
    pp = rect.mm_to_px(p)
    sx = (pp[:, 0:1] + n_in[:, 0:1] * offs[None, :]).astype(np.float32)
    sy = (pp[:, 1:2] + n_in[:, 1:2] * offs[None, :]).astype(np.float32)
    prof = cv2.remap(feat_f32, sx, sy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    inside = np.median(prof[:, offs >= 2.5], axis=1)
    outside = np.median(prof[:, offs <= -2.5], axis=1)
    mid = 0.5 * (inside + outside)
    d = prof - mid[:, None]
    up = (d[:, :-1] < 0) & (d[:, 1:] >= 0)
    near = np.abs(offs[:-1] + 0.25) <= max_shift_px
    cand = up & near[None, :]
    dist_to_0 = np.where(cand, np.abs(offs[:-1] + 0.25)[None, :], np.inf)
    j = np.argmin(dist_to_0, axis=1)
    has = np.isfinite(dist_to_0[np.arange(len(j)), j])
    d0, d1 = d[np.arange(len(j)), j], d[np.arange(len(j)), j + 1]
    frac = np.where(d1 != d0, -d0 / np.where(d1 != d0, d1 - d0, 1.0), 0.5)
    delta = offs[j] + 0.5 * frac
    margin = reach_px + 1.0
    inside_img = ((pp[:, 0] > margin) & (pp[:, 0] < rect.w - 1 - margin) &
                  (pp[:, 1] > margin) & (pp[:, 1] < rect.h - 1 - margin))
    ok = has & ((inside - outside) >= min_contrast) & inside_img
    delta = np.where(ok, np.clip(delta, -max_shift_px, max_shift_px), 0.0)
    new = rect.px_to_mm(pp + delta[:, None] * n_in)
    return new, ok, delta, inside_img


# ---------------------------------------------------------------------------
# Mask -> geometry (the heart of the measurement)
# ---------------------------------------------------------------------------
def mask_to_geometry(mask: np.ndarray, feat: np.ndarray, rect: Rectifier, cfg: VisionConfig):
    """Largest material blob as (outline, [holes]) in mm, the size ratio of the runner-up blob,
    and edge statistics.

    RETR_CCOMP gives a two-level hierarchy: outer borders and the holes inside them, from ONE
    segmentation. So cut-outs near the sheet edge are still found, and a cut-out that breaks the
    edge simply becomes part of the outline - no erosion margin that hides holes near the edge.

    Border following runs through the centres of the boundary pixels, i.e. half a pixel inside
    the true edge. A +0.5 px mitre buffer of the sheet AND of each void removes that bias; then
    every ring is refined to the sub-pixel 50 % crossing of the local intensity step."""
    contours, hier = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    if not contours:
        raise VisionError("Sheet boundary could not be detected. Improve lighting or reposition the sheet.")
    hier = hier[0]
    outers = [i for i in range(len(contours)) if hier[i][3] < 0]
    areas = np.array([cv2.contourArea(contours[i]) for i in outers]) * rect.s ** 2
    order = np.argsort(areas)[::-1]
    best = outers[int(order[0])]
    if areas[order[0]] < cfg.min_sheet_area_mm2:
        raise VisionError("Sheet boundary could not be detected (largest region is only "
                          f"{areas[order[0]] / 1e6:.3f} m2). Improve lighting or reposition the sheet.")
    runner_up = float(areas[order[1]] / areas[order[0]]) if len(order) > 1 else 0.0

    # Holes are traced as OUTER contours of the voids inside the sheet: the hole border that
    # findContours gives for the material follows material pixels 8-connected and chamfers every
    # hole corner. Traced on the void's own pixels, both sides get the same +0.5 px treatment.
    half = 0.5 * rect.s
    filled = np.zeros_like(mask)
    cv2.drawContours(filled, contours, best, 255, thickness=cv2.FILLED)
    voids = cv2.bitwise_and(filled, cv2.bitwise_not(mask))
    void_cs, _ = cv2.findContours(voids, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    def px_poly(c):
        c = c.reshape(-1, 2)
        if len(c) < 3:                                   # 1-2 px speck: use its pixel box
            x, y, w, h = cv2.boundingRect(c.reshape(-1, 1, 2))
            c = np.array([[x, y], [x + w - 1, y], [x + w - 1, y + h - 1], [x, y + h - 1]], float)
        return make_valid(Polygon(rect.px_to_mm(c))).buffer(half, join_style="mitre", mitre_limit=2.0)

    shell = px_poly(contours[best])
    geom = shell.difference(unary_union([px_poly(c) for c in void_cs])) if void_cs else shell

    feat_f32 = feat.astype(np.float32)
    parts, oks, shifts = [], [], []
    for p in _polygons(geom):
        p = orient(p, 1.0)       # exterior CCW, holes CW: left normal -> material
        ext, ok, dl, inner = refine_ring(p.exterior.coords, feat_f32, rect)
        oks.append(ok[inner]); shifts.append(dl[ok])
        holes = []
        for r in p.interiors:
            h, ok, dl, inner = refine_ring(r.coords, feat_f32, rect)
            oks.append(ok[inner]); shifts.append(dl[ok])
            holes.append(h)
        parts.extend(_polygons(make_valid(Polygon(ext, holes))))
    # Geometric opening (square element, keeps corners): removes spikes narrower than the minimum
    # feature - e.g. where a bright slat top meets the sheet edge. Such spikes claim material that
    # does not exist, which is the unsafe direction for nesting.
    r_open = 0.5 * cfg.min_feature_mm
    geom = unary_union(parts).intersection(rect.bed_box)
    geom = geom.buffer(-r_open, join_style="mitre", mitre_limit=5.0).buffer(r_open, join_style="mitre", mitre_limit=5.0)
    geom = _as_area(make_valid(geom.intersection(rect.bed_box)).simplify(cfg.simplify_mm, preserve_topology=True))
    polys = _polygons(geom)
    if not polys:
        raise VisionError("Sheet boundary could not be detected. Improve lighting or reposition the sheet.")
    outline = _as_area(unary_union([Polygon(p.exterior) for p in polys]))
    holes = [Polygon(r) for p in polys for r in p.interiors]
    ok_all = np.concatenate(oks) if oks else np.zeros(0, bool)
    sh = np.concatenate(shifts) if shifts else np.zeros(0)
    stats = {"edge_support": float(ok_all.mean()) if len(ok_all) else 0.0,
             "refine_shift_rms_px": float(np.sqrt(np.mean(sh ** 2))) if len(sh) else 0.0,
             "boundary_samples": int(len(ok_all))}
    return outline, holes, runner_up, stats


def _robust_rect_dims(outline, step: float) -> dict[str, Any]:
    """Length/width of the sheet from the minimum rotated rectangle's orientation, but with each
    side placed at the MEDIAN of the boundary points near it. Plain minAreaRect sits on the
    outermost pixel staircase and over-reports rotated sheets by up to a pixel per side."""
    hull = outline.convex_hull
    mrr = hull.minimum_rotated_rectangle
    c = np.asarray(mrr.exterior.coords)[:4]
    e1, e2 = c[1] - c[0], c[2] - c[1]
    long_edge = e1 if np.hypot(*e1) >= np.hypot(*e2) else e2
    theta = math.atan2(long_edge[1], long_edge[0])
    if theta > math.pi / 2:
        theta -= math.pi
    elif theta <= -math.pi / 2:
        theta += math.pi
    rot = np.array([[math.cos(theta), math.sin(theta)], [-math.sin(theta), math.cos(theta)]])
    pts = np.vstack([_ring_samples(p.exterior, step) for p in _polygons(outline)])
    q = (pts - pts.mean(axis=0)) @ rot.T
    ends = []
    for axis in (0, 1):
        lo, hi = q[:, axis].min(), q[:, axis].max()
        band = max(3.0 * step, 0.03 * (hi - lo))
        ends.append((np.median(q[q[:, axis] <= lo + band, axis]), np.median(q[q[:, axis] >= hi - band, axis])))
    length, width = ends[0][1] - ends[0][0], ends[1][1] - ends[1][0]
    minx, miny, maxx, maxy = outline.bounds
    return {"length_mm": float(length), "width_mm": float(width), "angle_deg": float(math.degrees(theta)),
            "bbox_mm": [float(minx), float(miny), float(maxx), float(maxy)],
            "rectangularity": float(outline.area / mrr.area) if mrr.area > 0 else 0.0}


def classify_cutout(poly: Polygon, cfg: VisionConfig) -> dict[str, Any]:
    """circle / rectangle / irregular, from geometry alone (no Hough accumulator to tune)."""
    area = poly.area
    pts = _ring_samples(poly.exterior, 0.5)
    cx, cy, r, rms = fit_circle(pts)
    info: dict[str, Any] = {"area_mm2": float(area), "perimeter_mm": float(poly.length),
                            "centroid_mm": [float(poly.centroid.x), float(poly.centroid.y)]}
    tol = max(cfg.circle_rms_mm, cfg.circle_rms_frac * r)
    if r > 0 and rms <= tol and abs(area / (math.pi * r * r) - 1.0) < 0.04:
        info.update(type="circle", center_mm=[cx, cy], radius_mm=r, diameter_mm=2 * r, fit_rms_mm=rms)
        return info
    mrr = poly.minimum_rotated_rectangle
    if mrr.area > 0 and area / mrr.area >= cfg.rect_fill:
        d = _robust_rect_dims(poly, 0.25)
        info.update(type="rectangle", length_mm=d["length_mm"], width_mm=d["width_mm"], angle_deg=d["angle_deg"])
        return info
    minx, miny, maxx, maxy = poly.bounds
    info.update(type="irregular", bbox_mm=[float(minx), float(miny), float(maxx), float(maxy)])
    return info


def _edge_notches(outline, cfg: VisionConfig) -> list[Polygon]:
    """Material missing from the sheet's bounding rectangle (edge notches, L-shaped remnants).
    Report-only: the outline already carries the exact geometry."""
    r = cfg.min_feature_mm / 2.0
    missing = outline.convex_hull.minimum_rotated_rectangle.difference(outline)
    missing = missing.buffer(-r, join_style="mitre").buffer(r, join_style="mitre")
    return [p for p in _polygons(missing) if p.area >= cfg.edge_notch_min_area_mm2]


def _touching_sides(outline, rect: Rectifier) -> list[str]:
    """Bed sides the sheet outline runs along (>= 30 mm). There the true sheet edge may lie
    beyond the calibrated bed, so the measured edge is only the bed limit."""
    pts = np.vstack([_ring_samples(p.exterior, rect.s) for p in _polygons(outline)])
    W, H = rect.calib.bed_w_mm, rect.calib.bed_h_mm
    near = 1.5 * rect.s
    sides = {"top": pts[:, 1] < near, "bottom": pts[:, 1] > H - near,
             "left": pts[:, 0] < near, "right": pts[:, 0] > W - near}
    return [k for k, m in sides.items() if m.sum() * rect.s > 30.0]


# ---------------------------------------------------------------------------
# Result
# ---------------------------------------------------------------------------
@dataclass
class ScanResult:
    bed_w_mm: float
    bed_h_mm: float
    mm_per_px: float
    outline: Any                   # Polygon | MultiPolygon: outer boundary of the physical sheet
    material: Any                  # Polygon | MultiPolygon: sheet minus every existing cut-out
    cutouts: list[dict[str, Any]]
    edge_notches: list[dict[str, Any]]
    dims: dict[str, Any]
    confidence: float
    quality: dict[str, Any]
    warnings: list[str]
    segmentation: dict[str, Any]
    ortho: np.ndarray | None = None
    mask: np.ndarray | None = None
    corrected_by_operator: bool = False

    @property
    def sheet_area_mm2(self) -> float:
        return float(self.outline.area)

    @property
    def available_area_mm2(self) -> float:
        return float(self.material.area)

    @property
    def removed_area_mm2(self) -> float:
        return float(self.outline.area - self.material.area)

    @property
    def needs_confirmation(self) -> bool:
        return self.confidence < self.quality.get("confirm_below", 0.85) or bool(self.warnings)

    def to_dict(self, include_geometry: bool = True) -> dict[str, Any]:
        def ring(g):
            return np.round(np.asarray(g.exterior.coords), 2).tolist()

        def clean(d):
            out = {k: (round(v, 2) if isinstance(v, float) else
                       [round(x, 2) for x in v] if isinstance(v, list) and v and isinstance(v[0], float) else v)
                   for k, v in d.items() if k != "geometry"}
            if include_geometry:
                out["polygon_mm"] = ring(d["geometry"])
            return out

        sheet = {"length_mm": round(self.dims["length_mm"], 1), "width_mm": round(self.dims["width_mm"], 1),
                 "angle_deg": round(self.dims["angle_deg"], 2), "area_mm2": round(self.sheet_area_mm2, 0),
                 "bbox_mm": [round(v, 1) for v in self.dims["bbox_mm"]],
                 "rectangularity": round(self.dims["rectangularity"], 4)}
        if include_geometry:
            sheet["polygon_mm"] = [ring(p) for p in _polygons(self.outline)]
        return {
            "bed": {"width_mm": self.bed_w_mm, "height_mm": self.bed_h_mm},
            "sheet": sheet,
            "cutouts": [clean(c) for c in self.cutouts],
            "edge_notches": [clean(n) for n in self.edge_notches],
            "material": {"sheet_area_mm2": round(self.sheet_area_mm2, 0),
                         "removed_area_mm2": round(self.removed_area_mm2, 0),
                         "available_area_mm2": round(self.available_area_mm2, 0),
                         "available_percent": round(100 * self.available_area_mm2 / self.sheet_area_mm2, 2),
                         **({"geometry_wkt": self.material.wkt} if include_geometry else {})},
            "scale_mm_per_px": self.mm_per_px,
            "confidence": round(self.confidence, 3),
            "needs_confirmation": self.needs_confirmation,
            "quality": self.quality,
            "segmentation": self.segmentation,
            "warnings": self.warnings,
            "corrected_by_operator": self.corrected_by_operator,
        }


def build_result(outline, holes: list[Polygon], bed_w_mm: float, bed_h_mm: float, mm_per_px: float,
                 cfg: VisionConfig, **extra) -> ScanResult:
    """Single source of truth for every derived number. material = sheet - union(cut-outs):
    geometry, not arithmetic, so overlapping or edge-breaking cut-outs are handled exactly."""
    bed = box(0.0, 0.0, bed_w_mm, bed_h_mm)
    outline = _as_area(make_valid(outline).intersection(bed))
    holes = [h for h in (_as_area(make_valid(h)) for h in holes) if not h.is_empty]
    material = _as_area(outline.difference(unary_union(holes))) if holes else outline
    inner_holes = [Polygon(r) for p in _polygons(material) for r in p.interiors]
    outline = _as_area(unary_union([Polygon(p.exterior) for p in _polygons(material)]))
    cutouts = []
    for i, h in enumerate(sorted(inner_holes, key=lambda g: (round(g.centroid.y, -1), g.centroid.x))):
        c = classify_cutout(h, cfg)
        c.update(id=f"cutout_{i + 1:03d}", geometry=h)
        cutouts.append(c)
    notches = [{"id": f"edge_{i + 1:03d}", "area_mm2": float(n.area),
                "centroid_mm": [float(n.centroid.x), float(n.centroid.y)], "geometry": n}
               for i, n in enumerate(_edge_notches(outline, cfg))]
    dims = _robust_rect_dims(outline, 0.5 * mm_per_px)
    return ScanResult(bed_w_mm, bed_h_mm, mm_per_px, outline, material, cutouts, notches, dims,
                      extra.pop("confidence", 1.0), extra.pop("quality", {}), extra.pop("warnings", []),
                      extra.pop("segmentation", {}), **extra)


def scan_sheet(image: np.ndarray, calib: BedCalibration, cfg: VisionConfig | None = None,
               reference_image: np.ndarray | None = None, rectifier: Rectifier | None = None) -> ScanResult:
    """Camera frame -> measured material map (all geometry in bed mm)."""
    cfg = cfg or VisionConfig()
    rect = rectifier or Rectifier(calib, cfg.mm_per_px)
    ortho = rect.warp(image)
    ref = rect.warp(reference_image) if reference_image is not None else None
    mask, feat, seg = segment_material(ortho, rect, cfg, ref)
    outline, holes, runner_up, edge = mask_to_geometry(mask, feat, rect, cfg)

    warnings = []
    if calib.rms_mm is not None and calib.rms_mm > 1.5:
        warnings.append(f"Calibration residual is {calib.rms_mm:.2f} mm - recalibrate before cutting.")
    biggest_hole = max((h.area for h in holes), default=0.0)
    if biggest_hole > cfg.max_hole_fraction * outline.area:
        raise VisionError("The detected region looks like the bed around a sheet, not a sheet. "
                          "Check the material polarity setting or capture an empty-bed reference.")
    support, touching = edge["edge_support"], _touching_sides(outline, rect)
    sep = seg["separability"]
    ambiguous = runner_up > cfg.ambiguity_ratio
    conf = support * min(1.0, sep / 0.75) * (0.5 if ambiguous else 1.0)
    if ambiguous:
        warnings.append(f"Multiple possible sheets: a second region is {runner_up:.0%} of the largest. "
                        "Only the largest is used.")
    if touching:
        warnings.append(f"Sheet touches the calibrated bed edge ({', '.join(touching)}): the real sheet edge "
                        "there may lie outside the camera's calibrated area. Confirm those edges.")
    if not rect.valid.all():
        lost = outline.intersection(_invalid_region(rect)).area
        if lost > 1.0:
            warnings.append("Part of the sheet is outside the camera view.")
    if sep < 0.5:
        warnings.append("Low contrast between sheet and bed. Improve lighting.")
    quality = {"edge_support": round(support, 3), "refine_shift_rms_px": round(edge["refine_shift_rms_px"], 3),
               "separability": sep, "runner_up_ratio": round(runner_up, 3),
               "calibration_rms_mm": calib.rms_mm, "calibration_points": calib.n_points,
               "confirm_below": cfg.confirm_below,
               "note": "Heuristic quality score, not a probability. Validate on real machine images."}
    return build_result(outline, holes, calib.bed_w_mm, calib.bed_h_mm, rect.s, cfg,
                        confidence=conf, quality=quality, warnings=warnings, segmentation=seg,
                        ortho=ortho, mask=mask)


def _invalid_region(rect: Rectifier):
    m = (~rect.valid).astype(np.uint8) * 255
    cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return unary_union([Polygon(rect.px_to_mm(c.reshape(-1, 2))).buffer(rect.s) for c in cs if len(c) >= 3])


def apply_corrections(result: ScanResult, cfg: VisionConfig | None = None, remove_ids: Sequence[str] = (),
                      add_cutouts_mm: Sequence[Any] = (), sheet_polygon_mm=None) -> ScanResult:
    """Operator 'Confirm / Adjust': drop false cut-outs, add missed ones, replace the sheet outline.
    Added cut-outs may be polygons (list of [x, y]) or circles ({'center': [x, y], 'radius': r})."""
    cfg = cfg or VisionConfig()
    holes = [c["geometry"] for c in result.cutouts if c["id"] not in set(remove_ids)]
    for a in add_cutouts_mm:
        holes.append(Point(a["center"]).buffer(a["radius"], quad_segs=64) if isinstance(a, dict)
                     else Polygon(_pts(a)))
    outline = Polygon(_pts(sheet_polygon_mm)) if sheet_polygon_mm is not None else result.outline
    return build_result(outline, holes, result.bed_w_mm, result.bed_h_mm, result.mm_per_px, cfg,
                        confidence=result.confidence, quality=result.quality,
                        warnings=[w for w in result.warnings], segmentation=result.segmentation,
                        ortho=result.ortho, mask=result.mask, corrected_by_operator=True)


def usable_region(result: ScanResult, uncertainty_mm: float = 3.0, edge_margin_mm: float = 0.0):
    """The region the nesting engine may use: material shrunk by the measurement uncertainty plus
    the required part-to-edge margin. Shrinking the MATERIAL (not the parts) applies it both to
    the sheet edge and to every existing cut-out in one exact operation."""
    d = uncertainty_mm + edge_margin_mm
    return _as_area(result.material.buffer(-d, join_style="mitre", mitre_limit=5.0)) if d > 0 else result.material


def validate_scan(result: ScanResult) -> list[str]:
    """Checks that must pass before anything is exported."""
    problems = []
    bed = box(0.0, 0.0, result.bed_w_mm, result.bed_h_mm).buffer(1e-6)
    if result.material.is_empty or result.available_area_mm2 <= 0:
        problems.append("No usable material.")
    if not result.material.is_valid:
        problems.append("Material geometry is invalid.")
    if not bed.contains(result.outline):
        problems.append("Sheet outline extends beyond the machine bed.")
    for c in result.cutouts:
        if not result.outline.buffer(1e-6).contains(c["geometry"]):
            problems.append(f"{c['id']} lies outside the sheet.")
    return problems


# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------
def draw_blueprint(result: ScanResult, rect: Rectifier, show_hud: bool = True) -> np.ndarray:
    vis = result.ortho.copy()
    t = max(1, int(round(min(rect.w, rect.h) / 400)))
    fs = max(0.35, min(rect.w, rect.h) / 1600)
    SH = 4                                                  # sub-pixel drawing (1/16 px)

    def ipts(coords):
        return np.round((rect.mm_to_px(coords)) * (1 << SH)).astype(np.int32).reshape(-1, 1, 2)

    def put(txt, xy, color, scale=fs, th=1):
        x, y = int(xy[0]), int(xy[1])
        cv2.putText(vis, txt, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), th + 2, cv2.LINE_AA)
        cv2.putText(vis, txt, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, th, cv2.LINE_AA)

    overlay = vis.copy()
    for p in _polygons(result.material):
        cv2.fillPoly(overlay, [ipts(p.exterior.coords)], (60, 170, 60), cv2.LINE_AA, SH)
        for r in p.interiors:
            cv2.fillPoly(overlay, [ipts(r.coords)], (40, 40, 200), cv2.LINE_AA, SH)
    cv2.addWeighted(overlay, 0.28, vis, 0.72, 0, vis)
    for n in result.edge_notches:
        cv2.polylines(vis, [ipts(n["geometry"].exterior.coords)], True, (0, 165, 255), t, cv2.LINE_AA, SH)
    for p in _polygons(result.outline):
        cv2.polylines(vis, [ipts(p.exterior.coords)], True, (0, 255, 0), t + 1, cv2.LINE_AA, SH)
    for c in result.cutouts:
        cv2.polylines(vis, [ipts(c["geometry"].exterior.coords)], True, (0, 0, 255), t, cv2.LINE_AA, SH)
        cx, cy = rect.mm_to_px([c["centroid_mm"]])[0]
        lbl = c["id"].replace("cutout_", "#")
        if c["type"] == "circle":
            lbl += f" D{c['diameter_mm']:.1f}"
        elif c["type"] == "rectangle":
            lbl += f" {c['length_mm']:.0f}x{c['width_mm']:.0f}"
        put(lbl, (cx - 30 * fs, cy - 8 * fs), (0, 230, 255), fs * 0.9)
    minx, miny, maxx, maxy = result.dims["bbox_mm"]
    put(f"{result.dims['length_mm']:.1f} x {result.dims['width_mm']:.1f} mm",
        rect.mm_to_px([[minx, miny]])[0] + [8, 22 * fs + 8], (0, 255, 255), fs * 1.3, t + 1)
    if show_hud:
        lines = ["BANSALI SMARTNEST - MEASURED MATERIAL MAP",
                 f"Sheet {result.dims['length_mm']:.1f} x {result.dims['width_mm']:.1f} mm  "
                 f"(angle {result.dims['angle_deg']:+.2f} deg)",
                 f"Sheet area      {result.sheet_area_mm2 / 1e6:.4f} m2",
                 f"Cut-outs        {len(result.cutouts)}  ({result.removed_area_mm2 / 1e6:.4f} m2 removed)",
                 f"Edge notches    {len(result.edge_notches)}",
                 f"Available       {result.available_area_mm2 / 1e6:.4f} m2",
                 f"Quality score   {result.confidence:.2f}" + ("  -> CONFIRM" if result.needs_confirmation else "")]
        lh = int(26 * fs * 1.6)
        bw, bh = int(560 * fs * 1.6), lh * len(lines) + 16
        ov = vis.copy()
        cv2.rectangle(ov, (10, 10), (10 + bw, 10 + bh), (20, 24, 30), -1)
        cv2.addWeighted(ov, 0.82, vis, 0.18, 0, vis)
        for i, line in enumerate(lines):
            put(line, (20, 10 + lh * (i + 1)), (0, 240, 255) if i == 0 else (235, 240, 245), fs * 1.05)
    return vis


# ---------------------------------------------------------------------------
# DXF export (machine millimetres, Y up) + read-back validation
# ---------------------------------------------------------------------------
def to_cad(pts, bed_h_mm: float, origin: str = "bottom-left") -> np.ndarray:
    """Bed mm (image-like, y down) -> CAD mm (right-handed, Y up).
    Exporting image-style y-down coordinates would MIRROR the whole layout on the machine."""
    p = _pts(pts)
    if origin == "bottom-left":
        return np.c_[p[:, 0], bed_h_mm - p[:, 1]]
    if origin == "top-left":
        return np.c_[p[:, 0], -p[:, 1]]
    raise ValueError("origin must be 'bottom-left' or 'top-left'")


def export_dxf(result: ScanResult, path: str, origin: str = "bottom-left", include_bed: bool = True) -> dict:
    import ezdxf

    problems = validate_scan(result)
    if problems:
        raise VisionError("Export blocked: " + "; ".join(problems))
    doc = ezdxf.new("R2010", setup=True)
    doc.units = ezdxf.units.MM
    doc.header["$MEASUREMENT"] = 1
    for name, color in (("BED", 8), ("SHEET_BOUNDARY", 3), ("EXISTING_CUTOUTS", 1)):
        doc.layers.add(name, color=color)
    msp = doc.modelspace()
    H = result.bed_h_mm

    def poly(coords, layer):
        msp.add_lwpolyline([tuple(p) for p in to_cad(np.asarray(coords)[:-1], H, origin)],
                           close=True, dxfattribs={"layer": layer})

    if include_bed:
        poly(box(0, 0, result.bed_w_mm, H).exterior.coords, "BED")
    for p in _polygons(result.outline):
        poly(p.exterior.coords, "SHEET_BOUNDARY")
    n_circles = 0
    for c in result.cutouts:
        if c["type"] == "circle":
            (X, Y), = to_cad([c["center_mm"]], H, origin)
            msp.add_circle((float(X), float(Y)), float(c["radius_mm"]), dxfattribs={"layer": "EXISTING_CUTOUTS"})
            n_circles += 1
        else:
            poly(c["geometry"].exterior.coords, "EXISTING_CUTOUTS")
    doc.saveas(path)
    return {"path": path, "sheet_outlines": len(_polygons(result.outline)), "cutouts": len(result.cutouts),
            "circles": n_circles, "origin": origin, "units": "mm"}


def read_dxf_material(path: str, bed_h_mm: float, origin: str = "bottom-left"):
    """Rebuild the material geometry from an exported DXF (back in bed mm)."""
    import ezdxf

    msp = ezdxf.readfile(path).modelspace()

    def back(pts):
        p = _pts(pts)
        return np.c_[p[:, 0], bed_h_mm - p[:, 1]] if origin == "bottom-left" else np.c_[p[:, 0], -p[:, 1]]

    sheets, holes = [], []
    for e in msp.query("LWPOLYLINE"):
        g = Polygon(back([(x, y) for x, y, *_ in e.get_points()]))
        (sheets if e.dxf.layer == "SHEET_BOUNDARY" else holes if e.dxf.layer == "EXISTING_CUTOUTS" else []).append(g)
    for e in msp.query("CIRCLE"):
        (c,) = back([(e.dxf.center.x, e.dxf.center.y)])
        holes.append(Point(c).buffer(e.dxf.radius, quad_segs=128))
    outline = unary_union(sheets)
    return outline.difference(unary_union(holes)) if holes else outline


def validate_dxf(path: str, result: ScanResult, origin: str = "bottom-left", tol_mm2: float | None = None) -> dict:
    """Re-read the file and compare it to the scan: the exported geometry is validated again."""
    got = read_dxf_material(path, result.bed_h_mm, origin)
    diff = got.symmetric_difference(result.material).area
    tol = tol_mm2 if tol_mm2 is not None else 1e-3 * result.available_area_mm2
    return {"ok": bool(diff <= tol), "dxf_area_mm2": got.area, "scan_area_mm2": result.available_area_mm2,
            "symmetric_difference_mm2": diff, "tolerance_mm2": tol}

In [ ]:
%%writefile smartnest_synthetic.py
"""
Synthetic laser-bed scenes with exactly known geometry, for demo mode and automated tests.

Each scene is defined in bed millimetres, rendered as a flat texture at 0.5 mm/px (anti-aliased),
then photographed by a physically posed pinhole camera (tilt, pan, roll, optional barrel
distortion) with 2x supersampling, vignetting, sensor noise and JPEG compression. The ground
truth (sheet outline, every cut-out, the usable material) is kept as Shapely geometry, so
detection errors can be measured in mm and mm^2 instead of judged by eye.

It is still a simulation: real sheets add glare, dirt, burrs, dross, bent edges and thickness
parallax. Passing here is necessary, not sufficient - validate on real Bansali machine images.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Any, Callable

import cv2
import numpy as np
from shapely import affinity
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union

from smartnest_vision import _polygons, apply_homography, aruco_layout, distort_pixels, undistort_pixels


# ---------------------------------------------------------------------------
# Camera
# ---------------------------------------------------------------------------
@dataclass
class Camera:
    K: np.ndarray
    dist: np.ndarray | None
    R: np.ndarray                 # world (bed mm, z into the bed) -> camera
    t: np.ndarray
    size: tuple[int, int]         # (width, height)

    @property
    def H_mm_to_img(self) -> np.ndarray:
        H = self.K @ np.c_[self.R[:, 0], self.R[:, 1], self.t]
        return H / H[2, 2]

    def project(self, pts_mm) -> np.ndarray:
        p = apply_homography(self.H_mm_to_img, pts_mm)
        return distort_pixels(p, self.K, self.dist) if self.dist is not None else p


def make_camera(bed_w: float, bed_h: float, size=(1920, 1080), f_px: float = 1400.0, fill: float = 0.86,
                tilt_deg: float = 9.0, pan_deg: float = 0.8, roll_deg: float = 0.7, dist=None) -> Camera:
    """Camera looking at the bed centre from above and slightly in front (tilt), so the far edge
    of the bed appears shorter than the near edge - a real, not hand-drawn, perspective."""
    W, H = size
    D = f_px * bed_w / (fill * W)
    T = np.array([bed_w / 2, bed_h / 2, 0.0])
    th = math.radians(tilt_deg)
    C = T + D * np.array([0.0, math.sin(th), -math.cos(th)])
    z = (T - C) / np.linalg.norm(T - C)
    x = np.array([1.0, 0.0, 0.0])
    x = x - x.dot(z) * z
    x /= np.linalg.norm(x)
    y = np.cross(z, x)
    R = np.vstack([x, y, z])
    p, r = math.radians(pan_deg), math.radians(roll_deg)
    Ry = np.array([[math.cos(p), 0, math.sin(p)], [0, 1, 0], [-math.sin(p), 0, math.cos(p)]])
    Rz = np.array([[math.cos(r), -math.sin(r), 0], [math.sin(r), math.cos(r), 0], [0, 0, 1]])
    R = Rz @ Ry @ R
    K = np.array([[f_px, 0, (W - 1) / 2], [0, f_px, (H - 1) / 2], [0, 0, 1]], dtype=np.float64)
    return Camera(K, None if dist is None else np.asarray(dist, dtype=np.float64), R, -R @ C, (W, H))


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------
@dataclass
class BedStyle:
    gap: tuple = (38, 40, 44)          # dark space between slats (BGR)
    slat: tuple = (112, 116, 122)      # slat tops
    slat_pitch_mm: float = 60.0
    slat_width_mm: float = 3.0
    frame: tuple = (58, 62, 68)
    rail: tuple = (96, 102, 108)


def _fill(img, polys_mm, value, mm2tex, shift=4):
    for p in polys_mm:
        pts = np.round(mm2tex(np.asarray(p.exterior.coords)) * (1 << shift)).astype(np.int32)
        cv2.fillPoly(img, [pts.reshape(-1, 1, 2)], value, cv2.LINE_AA, shift)


def coverage_alpha(material, shape_hw, mm2tex, tex_res: float, ss: int = 4, tile: int = 1024) -> np.ndarray:
    """Unbiased anti-aliased mask: 255 * (fraction of each texel covered by material).

    cv2.fillPoly is NOT unbiased: LINE_8 includes every pixel the edge touches (+0.5 px on
    average) and LINE_AA pushes edges out by 0.3-0.8 px. Ground truth rendered that way would
    make every detector look 0.3 mm wrong. Here: 4x supersampled LINE_8 fill of polygons shrunk
    by exactly half a sub-pixel, then an exact box-filter downsample (residual <= 1/8 texel)."""
    th, tw = shape_hw
    alpha = np.zeros((th, tw), np.uint8)
    e = 0.5 * tex_res / ss
    shells = [Polygon(p.exterior).buffer(-e, join_style="mitre") for p in _polygons(material)]
    holes = [Polygon(r).buffer(-e, join_style="mitre") for p in _polygons(material) for r in p.interiors]
    if not shells:
        return alpha
    minx, miny, maxx, maxy = unary_union(shells).bounds
    (j_lo, i_lo), (j_hi, i_hi) = np.floor(mm2tex([[minx, miny]])[0]) - 2, np.ceil(mm2tex([[maxx, maxy]])[0]) + 2
    j_lo, i_lo, j_hi, i_hi = int(max(j_lo, 0)), int(max(i_lo, 0)), int(min(j_hi, tw)), int(min(i_hi, th))
    SH = 4
    for i0 in range(i_lo, i_hi, tile):
        for j0 in range(j_lo, j_hi, tile):
            h, w = min(tile, i_hi - i0), min(tile, j_hi - j0)
            hi = np.zeros((h * ss, w * ss), np.uint8)

            def draw(geoms, value):
                for g in _polygons(unary_union(geoms)) if geoms else []:
                    for ring in [g.exterior]:
                        t = mm2tex(np.asarray(ring.coords)) - [j0, i0]           # texel coords in tile
                        q = ss * (t + 0.5) - 0.5                                 # sub-pixel coords
                        cv2.fillPoly(hi, [np.round(q * (1 << SH)).astype(np.int32).reshape(-1, 1, 2)],
                                     value, cv2.LINE_8, SH)
            draw(shells, 255)
            draw(holes, 0)
            alpha[i0:i0 + h, j0:j0 + w] = cv2.resize(hi, (w, h), interpolation=cv2.INTER_AREA)
    return alpha


def render(bed_w: float, bed_h: float, material, cam: Camera, *, metal=(188, 192, 196),
           style: BedStyle | None = None, glare: bool = True, markers: dict[int, np.ndarray] | None = None,
           seed: int = 0, tex_res: float = 0.5, margin: float = 260.0, noise: float = 2.5,
           jpeg_quality: int = 92) -> np.ndarray:
    style = style or BedStyle()
    rng = np.random.default_rng(seed)
    r, m = tex_res, margin
    tw, th = int(math.ceil((bed_w + 2 * m) / r)), int(math.ceil((bed_h + 2 * m) / r))

    def mm2tex(p):
        return (np.asarray(p, dtype=np.float64) + m) / r - 0.5

    tex = np.empty((th, tw, 3), np.uint8)
    tex[:] = style.frame
    _fill(tex, [box(-40, -40, bed_w + 40, bed_h + 40)], style.rail, mm2tex)
    _fill(tex, [box(0, 0, bed_w, bed_h)], style.gap, mm2tex)
    x = style.slat_pitch_mm / 2
    while x < bed_w:
        jitter = int(rng.integers(-10, 11))
        col = tuple(int(np.clip(c + jitter, 0, 255)) for c in style.slat)
        _fill(tex, [box(x - style.slat_width_mm / 2, 0, x + style.slat_width_mm / 2, bed_h)], col, mm2tex)
        x += style.slat_pitch_mm
    if markers:
        d = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
        for mid, c in markers.items():
            x0, y0 = c[0]
            size = c[1][0] - c[0][0]
            _fill(tex, [box(x0 - 15, y0 - 15, x0 + size + 15, y0 + size + 15)], (245, 245, 245), mm2tex)
            n = int(round(size / r))
            img = cv2.aruco.generateImageMarker(d, int(mid), n)
            j0, i0 = int(round((x0 + m) / r)), int(round((y0 + m) / r))
            tex[i0:i0 + n, j0:j0 + n] = img[:, :, None]

    # sheet: anti-aliased alpha, brushed-metal texture, optional specular glare
    alpha = coverage_alpha(material, (th, tw), mm2tex, r)
    rows = cv2.GaussianBlur(rng.normal(0, 3.5, (th, 1)).astype(np.float32), (1, 0), sigmaX=0, sigmaY=6).ravel()
    if glare and not material.is_empty:
        c = material.representative_point()
        gx, gy = mm2tex([[c.x + 150, c.y - 80]])[0]
        sig = 240.0 / r
    band = 512
    yy_all = np.arange(th, dtype=np.float32)
    xx = np.arange(tw, dtype=np.float32)
    for i0 in range(0, th, band):
        i1 = min(th, i0 + band)
        a = alpha[i0:i1].astype(np.float32)[:, :, None] / 255.0
        met = np.empty((i1 - i0, tw, 3), np.float32)
        met[:] = np.asarray(metal, np.float32)
        met += rows[i0:i1, None, None]
        met += rng.normal(0, 1.5, (i1 - i0, tw, 1)).astype(np.float32)
        if glare and not material.is_empty:
            yy = yy_all[i0:i1, None]
            met += (55.0 * np.exp(-((xx[None, :] - gx) ** 2 + (yy - gy) ** 2) / (2 * sig * sig)))[:, :, None]
        seg = tex[i0:i1].astype(np.float32)
        tex[i0:i1] = np.clip(seg * (1 - a) + met * a, 0, 255).astype(np.uint8)
    tex = cv2.GaussianBlur(tex, (0, 0), 0.6)            # optics / pre-filter before sampling

    # photograph: 2x supersampled, one resampling from the texture
    W, H = cam.size
    T_tex2mm = np.array([[r, 0, 0.5 * r - m], [0, r, 0.5 * r - m], [0, 0, 1]])
    S2 = np.array([[2, 0, 0.5], [0, 2, 0.5], [0, 0, 1]], dtype=np.float64)
    if cam.dist is None:
        H2 = S2 @ cam.H_mm_to_img @ T_tex2mm
        big = cv2.warpPerspective(tex, H2, (2 * W, 2 * H), flags=cv2.INTER_LINEAR | cv2.WARP_FILL_OUTLIERS,
                                  borderValue=style.frame)
    else:
        # distortion is smooth: solve it on a 4 px grid, then let cv2.resize interpolate the map.
        # Grid node k sits at image x = 4k + 1.5, which is exactly where cv2.resize samples it.
        gw, gh = W // 4, H // 4
        u, v = np.meshgrid(4 * np.arange(gw) + 1.5, 4 * np.arange(gh) + 1.5)
        ideal = undistort_pixels(np.c_[u.ravel(), v.ravel()], cam.K, cam.dist)
        mm = apply_homography(np.linalg.inv(cam.H_mm_to_img), ideal)
        tp = mm2tex(mm).astype(np.float32)
        mx = cv2.resize(tp[:, 0].reshape(gh, gw), (2 * W, 2 * H), interpolation=cv2.INTER_CUBIC)
        my = cv2.resize(tp[:, 1].reshape(gh, gw), (2 * W, 2 * H), interpolation=cv2.INTER_CUBIC)
        big = cv2.remap(tex, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=style.frame)
    img = cv2.resize(big, (W, H), interpolation=cv2.INTER_AREA).astype(np.float32)

    # lighting falloff, sensor noise, compression
    u, v = np.meshgrid(np.linspace(-1, 1, W, dtype=np.float32), np.linspace(-1, 1, H, dtype=np.float32))
    gain = (1.0 - 0.18 * (u * u + v * v) / 2.0) * (1.0 + 0.06 * u)
    img = img * gain[:, :, None] + rng.normal(0, noise, img.shape).astype(np.float32)
    img = np.clip(img, 0, 255).astype(np.uint8)
    if jpeg_quality:
        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return img


# ---------------------------------------------------------------------------
# Scenarios
# ---------------------------------------------------------------------------
@dataclass
class Scene:
    name: str
    description: str
    bed_w: float
    bed_h: float
    camera: Camera
    image: np.ndarray
    empty_bed: np.ndarray
    corners_px: np.ndarray                    # true image position of bed corners TL, TR, BR, BL
    sheet: Polygon                            # true outer outline
    holes: list[dict[str, Any]]               # true interior cut-outs
    material: Any                             # true usable material
    notches: list[Polygon] = field(default_factory=list)
    markers_mm: dict[int, np.ndarray] = field(default_factory=dict)
    hint: dict[str, Any] = field(default_factory=dict)


def circle(cx, cy, r):
    return {"type": "circle", "center": (cx, cy), "radius": r, "geometry": Point(cx, cy).buffer(r, quad_segs=256)}


def rect(cx, cy, w, h, angle=0.0):
    g = affinity.rotate(box(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), angle, origin=(cx, cy))
    return {"type": "rectangle", "center": (cx, cy), "size": (w, h), "angle": angle, "geometry": g}


def poly(pts):
    g = Polygon(pts)
    return {"type": "irregular", "center": (g.centroid.x, g.centroid.y), "geometry": g}


def _marker_layout(bed_w, bed_h, cam: Camera, size=90.0) -> dict[int, np.ndarray]:
    centers, i = {}, 0
    for x in (-110.0, bed_w + 110.0):
        for fy in (0.12, 0.5, 0.88):
            centers[i] = (x, round(fy * bed_h))
            i += 1
    for x in (0.25 * bed_w, 0.75 * bed_w):
        for y in (-110.0, bed_h + 110.0):
            centers[i] = (round(x), y)
            i += 1
    layout = aruco_layout(centers, size)
    W, H = cam.size
    ok = {}
    for k, c in layout.items():
        p = cam.project(np.vstack([c, c.mean(axis=0) + [[-60, -60], [60, -60], [60, 60], [-60, 60]]]))
        if (p[:, 0] > 5).all() and (p[:, 0] < W - 6).all() and (p[:, 1] > 5).all() and (p[:, 1] < H - 6).all():
            ok[k] = c
    return ok


def _scene(name, description, bed_w, bed_h, sheet, holes, *, notches=(), cam=None, metal=(188, 192, 196),
           style=None, seed=0, hint=None, **render_kw) -> Scene:
    cam = cam or make_camera(bed_w, bed_h)
    hole_geoms = [h["geometry"] for h in holes]
    material = sheet.difference(unary_union(hole_geoms)) if holes else sheet
    markers = _marker_layout(bed_w, bed_h, cam)
    img = render(bed_w, bed_h, material, cam, metal=metal, style=style, markers=markers, seed=seed, **render_kw)
    empty = render(bed_w, bed_h, Polygon(), cam, style=style, markers=markers, seed=seed + 1000, **render_kw)
    corners = cam.project([[0, 0], [bed_w, 0], [bed_w, bed_h], [0, bed_h]])
    return Scene(name, description, bed_w, bed_h, cam, img, empty, corners, sheet, holes, material,
                 list(notches), markers, hint or {})


def scene_fresh_sheet(seed=0):
    sheet = box(180, 140, 2580, 1340)
    return _scene("fresh_sheet", "Fresh 2400 x 1200 mm sheet on a 3000 x 1500 mm bed, not centred.",
                  3000, 1500, sheet, [], seed=seed)


def scene_four_circles(seed=0):
    sheet = box(180, 140, 2580, 1340)
    holes = [circle(780, 480, 90), circle(1980, 480, 90), circle(780, 1000, 90), circle(1980, 1000, 90)]
    return _scene("four_circles", "Sheet with 4 pre-cut circular holes (D180 mm).", 3000, 1500, sheet, holes,
                  seed=seed)


def _mixed_holes():
    return [circle(600, 450, 90), circle(1100, 420, 40),
            circle(1500, 400, 12.5),                        # D25 bolt hole: 491 mm2
            rect(1950, 875, 300, 150), rect(900, 950, 220, 120, 30.0),
            circle(235, 1100, 35),                          # only 20 mm from the sheet edge
            poly([(1400, 900), (1600, 880), (1650, 1050), (1520, 1150), (1380, 1060)])]


def scene_mixed_holes(seed=0):
    sheet = box(180, 140, 2580, 1340)
    return _scene("mixed_holes", "Circles (incl. a D25 bolt hole), rectangles (one rotated 30 deg), an "
                  "irregular cut-out, and a hole 20 mm from the sheet edge.", 3000, 1500, sheet, _mixed_holes(),
                  seed=seed)


def scene_irregular_remnant(seed=0):
    base = box(0, 0, 2200, 1100)
    corner = box(2200 - 750, 0, 2200, 450)                 # L-shaped remnant
    semi = Point(900, 1100).buffer(90, quad_segs=256)       # half-round notch in the near edge
    local = base.difference(corner).difference(semi)
    place = lambda g: affinity.translate(affinity.rotate(g, 2.5, origin=(1100, 550)), 150, 210)
    sheet = place(local)
    holes = [circle(*place(Point(400, 300)).coords[0], 60), circle(*place(Point(1700, 800)).coords[0], 60),
             poly(list(place(Polygon([(700, 500), (950, 430), (1100, 600), (980, 800), (760, 760)])).exterior.coords))]
    notches = [place(corner), place(semi.intersection(base))]
    return _scene("irregular_remnant", "L-shaped remnant rotated 2.5 deg, with an edge notch, an irregular "
                  "cut-out and two holes.", 3000, 1500, sheet, holes, notches=notches, seed=seed)


def scene_dark_steel(seed=0):
    sheet = box(300, 200, 2300, 1200)
    holes = [circle(700, 500, 70), circle(1300, 500, 70), circle(1900, 500, 70), rect(1300, 900, 250, 180)]
    style = BedStyle(gap=(38, 40, 44), slat=(150, 148, 144), slat_pitch_mm=40.0, slat_width_mm=4.0)
    return _scene("dark_steel", "Dark mild-steel sheet on a slat bed with bright slat tops: brightness "
                  "alone cannot separate sheet from bed.", 3000, 1500, sheet, holes, metal=(84, 86, 90),
                  style=style, glare=False, seed=seed, hint={"needs_reference": True})


def scene_lens_distortion(seed=0):
    sheet = box(180, 140, 2580, 1340)
    cam = make_camera(3000, 1500, dist=[-0.20, 0.05, 0.0, 0.0, 0.0])
    return _scene("lens_distortion", "Same as mixed_holes, through a wide-angle lens with barrel distortion.",
                  3000, 1500, sheet, _mixed_holes(), cam=cam, seed=seed, hint={"needs_lens_model": True})


def scene_small_bed(seed=0):
    cam = make_camera(1500, 950, fill=0.8)
    sheet = box(120, 90, 1320, 890)
    holes = [circle(330, 280, 45), circle(1110, 280, 45), circle(330, 700, 45), circle(1110, 700, 45),
             rect(720, 490, 260, 40, 0.0)]
    return _scene("small_bed", "1500 x 950 mm bed (the Colab default) with a 1200 x 800 mm sheet: camera "
                  "resolution ~1 mm/px.", 1500, 950, sheet, holes, cam=cam, seed=seed)


SCENARIOS: dict[str, Callable[..., Scene]] = {
    "fresh_sheet": scene_fresh_sheet,
    "four_circles": scene_four_circles,
    "mixed_holes": scene_mixed_holes,
    "irregular_remnant": scene_irregular_remnant,
    "dark_steel": scene_dark_steel,
    "lens_distortion": scene_lens_distortion,
    "small_bed": scene_small_bed,
}


def make_scene(name: str, seed: int = 0) -> Scene:
    return SCENARIOS[name](seed=seed)

In [ ]:
%%writefile smartnest_cad.py
"""
Bansali SmartNest - CAD input/output for nesting.

    read_dxf_parts()      DXF -> validated closed part polygons (mm), quantities by de-duplication
    export_layout_dxf()   nested layout -> machine DXF (mm, Y up), original ARC/CIRCLE/SPLINE kept
    validate_layout_dxf() re-read the exported file and compare with the layout geometry
    make_sample_job_dxf() demo job with lines+arcs, bulged polylines, circles and holes

Invalid geometry is never repaired silently: open or self-intersecting contours are reported
with the part they belong to, and those contours are not nested.
"""
from __future__ import annotations

import math
import os
import re
from dataclasses import dataclass, field
from typing import Any

import numpy as np
from shapely import affinity
from shapely.geometry import LineString, Polygon
from shapely.geometry.polygon import orient
from shapely.ops import unary_union
from shapely.validation import explain_validity

MAX_UPLOAD_BYTES = 20 * 1024 * 1024
UNIT_SCALE = {0: 1.0, 1: 25.4, 2: 304.8, 4: 1.0, 5: 10.0, 6: 1000.0}   # $INSUNITS -> mm
SUPPORTED = {"LINE", "ARC", "CIRCLE", "LWPOLYLINE", "POLYLINE", "SPLINE", "ELLIPSE"}


@dataclass
class PartDef:
    name: str
    polygon: Polygon                        # mm, centred on its centroid (0, 0), holes included
    quantity: int = 1
    allowed_rotations: list[float] | None = None   # None = use the job's rotation step
    source_entities: list = field(default_factory=list)   # ezdxf entities (source file coordinates)
    source_offset: tuple[float, float] = (0.0, 0.0)       # centroid in source coordinates (mm)
    source_scale: float = 1.0                             # source units -> mm

    @property
    def area_mm2(self) -> float:
        return float(self.polygon.area)

    @property
    def cut_length_mm(self) -> float:
        return float(self.polygon.exterior.length + sum(r.length for r in self.polygon.interiors))

    def summary(self) -> dict[str, Any]:
        minx, miny, maxx, maxy = self.polygon.bounds
        return {"name": self.name, "quantity": self.quantity, "area_mm2": round(self.area_mm2, 1),
                "bbox_mm": [round(maxx - minx, 2), round(maxy - miny, 2)],
                "holes": len(self.polygon.interiors), "cut_length_mm": round(self.cut_length_mm, 1),
                "allowed_rotations": self.allowed_rotations}


@dataclass
class ImportReport:
    file: str
    units: str
    parts: list[dict[str, Any]] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    @property
    def ok(self) -> bool:
        return not self.errors


def safe_filename(name: str) -> str:
    base = os.path.basename(name or "upload.dxf")
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", base).strip("._") or "upload"
    return base[:120]


def _entities(msp):
    """Model-space entities with blocks (INSERT) exploded recursively."""
    for e in msp:
        if e.dxftype() == "INSERT":
            try:
                yield from _explode(e, depth=0)
            except Exception:  # noqa: BLE001
                continue
        else:
            yield e


def _explode(insert, depth):
    if depth > 8:
        return
    for v in insert.virtual_entities():
        if v.dxftype() == "INSERT":
            yield from _explode(v, depth + 1)
        else:
            yield v


def _to_points(e, scale: float, arc_tol: float):
    import ezdxf.path

    p = ezdxf.path.make_path(e)
    pts = np.array([(v.x, v.y) for v in p.flattening(distance=arc_tol / scale)], dtype=np.float64) * scale
    closed = bool(getattr(p, "is_closed", False))
    t = e.dxftype()
    if t == "CIRCLE" or (t == "LWPOLYLINE" and e.closed) or (t == "POLYLINE" and e.is_closed):
        closed = True
    if t == "SPLINE" and e.closed:
        closed = True
    if t == "ELLIPSE" and abs((e.dxf.end_param - e.dxf.start_param) - 2 * math.pi) < 1e-9:
        closed = True
    return pts, closed


def _chain(segments, tol):
    """Join open polylines end-to-end (within tol) into loops. Returns (loops, open_chains);
    each item is (points, [entity indices])."""
    segs = [(s, [i]) for i, s in enumerate(segments)]
    loops, open_chains = [], []
    used = [False] * len(segs)
    for i in range(len(segs)):
        if used[i]:
            continue
        used[i] = True
        pts, ids = segs[i][0].copy(), list(segs[i][1])
        grown = True
        while grown and np.hypot(*(pts[0] - pts[-1])) > tol:
            grown = False
            for j in range(len(segs)):
                if used[j]:
                    continue
                q = segs[j][0]
                if np.hypot(*(pts[-1] - q[0])) <= tol:
                    pts = np.vstack([pts, q[1:]])
                elif np.hypot(*(pts[-1] - q[-1])) <= tol:
                    pts = np.vstack([pts, q[::-1][1:]])
                elif np.hypot(*(pts[0] - q[-1])) <= tol:
                    pts = np.vstack([q[:-1], pts])
                elif np.hypot(*(pts[0] - q[0])) <= tol:
                    pts = np.vstack([q[::-1][:-1], pts])
                else:
                    continue
                used[j] = True
                ids += segs[j][1]
                grown = True
                break
        if np.hypot(*(pts[0] - pts[-1])) <= tol and len(pts) >= 4:
            loops.append((pts[:-1], ids))
        else:
            open_chains.append((pts, ids))
    return loops, open_chains


def _signature(poly: Polygon) -> tuple:
    mrr = poly.minimum_rotated_rectangle
    c = np.asarray(mrr.exterior.coords)[:4]
    a, b = sorted([np.hypot(*(c[1] - c[0])), np.hypot(*(c[2] - c[1]))])
    return (round(poly.area, 0), round(poly.exterior.length, 0), len(poly.interiors), round(a, 0), round(b, 0))


def read_dxf_parts(path: str, tol_mm: float = 0.05, arc_tol_mm: float = 0.05) -> tuple[list[PartDef], ImportReport]:
    """Every outer closed contour (with the closed contours inside it as holes) is one part.
    Identical parts drawn several times become one PartDef with that quantity."""
    import ezdxf
    from ezdxf import recover

    rep = ImportReport(file=os.path.basename(path), units="mm")
    if not path.lower().endswith(".dxf"):
        rep.errors.append("Only .dxf files are accepted for parts.")
        return [], rep
    if os.path.getsize(path) > MAX_UPLOAD_BYTES:
        rep.errors.append(f"File is larger than {MAX_UPLOAD_BYTES // (1024 * 1024)} MB.")
        return [], rep
    try:
        doc, auditor = recover.readfile(path)
    except (IOError, ezdxf.DXFStructureError) as e:
        rep.errors.append(f"Not a readable DXF file ({e}).")
        return [], rep
    code = int(doc.header.get("$INSUNITS", 0))
    scale = UNIT_SCALE.get(code)
    if scale is None:
        rep.errors.append(f"Unsupported drawing units ($INSUNITS={code}).")
        return [], rep
    rep.units = {0: "unitless (assumed mm)", 1: "inch", 2: "foot", 4: "mm", 5: "cm", 6: "m"}[code]
    if code == 0:
        rep.warnings.append("Drawing has no units set; millimetres assumed. Check the part sizes below.")

    ents, closed_loops, open_segs, open_ents, skipped = [], [], [], [], {}
    for e in _entities(doc.modelspace()):
        t = e.dxftype()
        if t not in SUPPORTED:
            skipped[t] = skipped.get(t, 0) + 1
            continue
        try:
            pts, closed = _to_points(e, scale, arc_tol_mm)
        except Exception as ex:  # noqa: BLE001
            rep.warnings.append(f"Could not read a {t} entity ({ex}).")
            continue
        if len(pts) < 2:
            continue
        ents.append(e)
        k = len(ents) - 1
        if closed:
            if np.hypot(*(pts[0] - pts[-1])) <= tol_mm:
                pts = pts[:-1]
            closed_loops.append((pts, [k]))
        else:
            open_segs.append(pts)
            open_ents.append(k)
    for t, n in skipped.items():
        if t not in ("TEXT", "MTEXT", "DIMENSION", "POINT", "HATCH", "LEADER", "MLEADER"):
            rep.warnings.append(f"Ignored {n} unsupported {t} entit{'y' if n == 1 else 'ies'}.")
    chained, open_chains = _chain(open_segs, tol_mm)
    loops = closed_loops + [(p, [open_ents[i] for i in ids]) for p, ids in chained]
    if not loops and not open_chains:
        rep.errors.append("No closed part contours found in the file.")
        return [], rep

    polys = []
    for pts, ids in loops:
        g = Polygon(pts)
        if not g.is_valid:
            polys.append((g, ids, "self-intersecting (" + explain_validity(g) + ")"))
        elif g.area < 1.0:
            polys.append((g, ids, "zero-area"))
        else:
            polys.append((orient(g), ids, None))
    order = sorted(range(len(polys)), key=lambda i: -abs(polys[i][0].area))
    parent = {}
    for pos, i in enumerate(order):
        pt = polys[i][0].representative_point() if polys[i][0].is_valid else polys[i][0].centroid
        cands = [j for j in order[:pos] if polys[j][2] is None and polys[j][0].contains(pt)]
        parent[i] = cands[-1] if cands else None
    depth = {}
    for i in order:
        depth[i] = 0 if parent[i] is None else depth[parent[i]] + 1

    raw_parts = []
    for i in order:
        if depth[i] % 2:
            continue
        g, ids, problem = polys[i]
        holes = [j for j in order if parent[j] == i]
        raw_parts.append((i, g, ids, problem, holes))
    raw_parts.sort(key=lambda t: (round(t[1].bounds[0], 3), round(t[1].bounds[1], 3)))   # drawing order
    for n, (i, g, ids, problem, holes) in enumerate(raw_parts):
        label = f"Part {n + 1:02d}"
        bad = [polys[j][2] for j in holes if polys[j][2]]
        if problem or bad:
            rep.errors.append(f"{label} has a {problem or bad[0]} contour and cannot be nested.")
    for pts, _ in open_chains:
        c = pts.mean(axis=0)
        owner = next((f"Part {n + 1:02d}" for n, (i, g, *_r) in enumerate(raw_parts)
                      if g.is_valid and g.buffer(1.0).intersects(LineString(pts))), None)
        where = f"{owner} contains" if owner else f"Near ({c[0]:.1f}, {c[1]:.1f}) mm there is"
        rep.errors.append(f"{where} an open contour (gap > {tol_mm} mm) and cannot be nested.")

    groups: dict[tuple, PartDef] = {}
    names = iter([chr(ord("A") + k) if k < 26 else f"P{k + 1}" for k in range(10_000)])
    bad_parts = {f"Part {n + 1:02d}" for n, _ in enumerate(raw_parts)
                 if any(f"Part {n + 1:02d} " in e for e in rep.errors)}
    for n, (i, g, ids, problem, holes) in enumerate(raw_parts):
        if f"Part {n + 1:02d}" in bad_parts:
            continue
        poly = orient(Polygon(g.exterior, [polys[j][0].exterior for j in holes]))
        c = poly.centroid
        centred = affinity.translate(poly, -c.x, -c.y)
        sig = _signature(centred)
        if sig in groups:
            groups[sig].quantity += 1
            continue
        src = [ents[k] for k in ids] + [ents[k] for j in holes for k in polys[j][1]]
        groups[sig] = PartDef(next(names), centred, 1, None, src, (c.x / scale, c.y / scale), scale)
    parts = list(groups.values())
    rep.parts = [p.summary() for p in parts]
    return parts, rep


# ---------------------------------------------------------------------------
# Layout export
# ---------------------------------------------------------------------------
def _matrix(part: PartDef, x: float, y: float, rot_deg: float):
    from ezdxf.math import Matrix44

    ox, oy = part.source_offset
    return (Matrix44.translate(-ox, -oy, 0) @ Matrix44.scale(part.source_scale, part.source_scale, 1)
            @ Matrix44.z_rotate(math.radians(rot_deg)) @ Matrix44.translate(x, y, 0))


def export_layout_dxf(nest, parts: list[PartDef], path: str, material=None, bed=None,
                      include_reference: bool = False, true_arcs: bool = True) -> dict[str, Any]:
    """Cutting file in machine millimetres (Y up, origin = bed bottom-left).
    Layer CUT holds only part contours. Reference layers (sheet, existing cut-outs, bed) are added
    only on request and use colour 8 - make sure your CAM does not cut them."""
    import ezdxf

    if not nest.export_enabled:
        raise ValueError("Export blocked: " + "; ".join(nest.failed_checks()))
    doc = ezdxf.new("R2010", setup=True)
    doc.units = ezdxf.units.MM
    doc.header["$MEASUREMENT"] = 1
    doc.layers.add("CUT", color=1)
    doc.layers.add("PART_ID", color=2)
    msp = doc.modelspace()
    arcs = polys = 0
    for pl in nest.placements:
        part = parts[pl.part_index]
        done = False
        if true_arcs and part.source_entities:
            try:                                  # transform detached copies; add only if all succeed
                m = _matrix(part, pl.x_mm, pl.y_mm, pl.rotation_deg)
                copies = []
                for e in part.source_entities:
                    ne = e.copy()
                    ne.transform(m)
                    ne.dxf.layer = "CUT"
                    copies.append(ne)
                for ne in copies:
                    msp.add_foreign_entity(ne, copy=True)
                    arcs += ne.dxftype() in ("ARC", "CIRCLE", "SPLINE", "ELLIPSE")
                done = True
            except Exception:  # noqa: BLE001 - fall back to the exact polygon contours
                done = False
        if not done:
            for ring in [pl.geometry.exterior, *pl.geometry.interiors]:
                msp.add_lwpolyline(list(ring.coords)[:-1], close=True, dxfattribs={"layer": "CUT"})
                polys += 1
        msp.add_text(pl.instance_id, height=min(20.0, 0.15 * math.sqrt(pl.geometry.area)),
                     dxfattribs={"layer": "PART_ID"}).set_placement((pl.x_mm, pl.y_mm))
    if include_reference and material is not None:
        doc.layers.add("REF_SHEET", color=8)
        doc.layers.add("REF_CUTOUTS", color=8)
        for g in getattr(material, "geoms", [material]):
            msp.add_lwpolyline(list(g.exterior.coords)[:-1], close=True, dxfattribs={"layer": "REF_SHEET"})
            for r in g.interiors:
                msp.add_lwpolyline(list(r.coords)[:-1], close=True, dxfattribs={"layer": "REF_CUTOUTS"})
    if include_reference and bed is not None:
        doc.layers.add("REF_BED", color=8)
        msp.add_lwpolyline([(0, 0), (bed[0], 0), (bed[0], bed[1]), (0, bed[1])], close=True,
                           dxfattribs={"layer": "REF_BED"})
    doc.saveas(path)
    return {"path": path, "parts": len(nest.placements), "true_curve_entities": arcs,
            "polyline_contours": polys, "units": "mm", "origin": "bed bottom-left, Y up"}


def validate_layout_dxf(path: str, nest, tol_mm2_per_part: float = 2.0) -> dict[str, Any]:
    """Re-read the CUT layer, rebuild the part polygons and compare with the layout."""
    import ezdxf

    msp = ezdxf.readfile(path).modelspace()
    segs, closed = [], []
    for e in msp.query('*[layer=="CUT"]'):
        if e.dxftype() not in SUPPORTED:
            continue
        pts, is_closed = _to_points(e, 1.0, 0.05)
        (closed if is_closed else segs).append(pts[:-1] if is_closed and np.hypot(*(pts[0] - pts[-1])) < 1e-6 else pts)
    loops, open_chains = _chain(segs, 0.05)
    rings = [Polygon(p) for p in closed] + [Polygon(p) for p, _ in loops]
    rings.sort(key=lambda g: -g.area)
    outers = []
    for g in rings:
        host = next((o for o in outers if o[0].contains(g.representative_point())), None)
        if host is None:
            outers.append([g, []])
        else:
            host[1].append(g)
    got = unary_union([o.difference(unary_union(h)) if h else o for o, h in outers])
    want = unary_union([pl.geometry for pl in nest.placements])
    diff = got.symmetric_difference(want).area
    tol = tol_mm2_per_part * max(1, len(nest.placements))
    return {"ok": bool(diff <= tol and not open_chains and len(outers) == len(nest.placements)),
            "contours_read": len(rings), "parts_read": len(outers), "open_contours": len(open_chains),
            "symmetric_difference_mm2": diff, "tolerance_mm2": tol}


def export_layout_svg(nest, material, bed, path: str) -> str:
    """Visual preview (not for cutting). SVG y runs down, so CAD Y is flipped."""
    W, H = bed

    def d(ring):
        pts = np.asarray(ring.coords)
        return "M " + " L ".join(f"{x:.2f},{H - y:.2f}" for x, y in pts) + " Z"

    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="-10 -10 {W + 20} {H + 20}" width="1200">',
           f'<rect x="0" y="0" width="{W}" height="{H}" fill="#1b1f24" stroke="#666" stroke-width="3"/>']
    for g in getattr(material, "geoms", [material]):
        out.append(f'<path d="{" ".join(d(r) for r in [g.exterior, *g.interiors])}" fill="#b9bec4" '
                   'fill-rule="evenodd" stroke="#2e7d32" stroke-width="3"/>')
    for pl in nest.placements:
        g = pl.geometry
        out.append(f'<path d="{" ".join(d(r) for r in [g.exterior, *g.interiors])}" fill="#1565c0" '
                   'fill-opacity="0.8" fill-rule="evenodd" stroke="#0d47a1" stroke-width="1.5"/>')
        out.append(f'<text x="{pl.x_mm:.1f}" y="{H - pl.y_mm:.1f}" font-size="18" fill="white" '
                   f'text-anchor="middle">{pl.instance_id}</text>')
    out.append("</svg>")
    with open(path, "w") as f:
        f.write("\n".join(out))
    return path


# ---------------------------------------------------------------------------
# Demo job
# ---------------------------------------------------------------------------
def make_sample_job_dxf(path: str) -> str:
    """Four part types, one of each, drawn with the entity mix real CAD files contain."""
    import ezdxf

    doc = ezdxf.new("R2010", setup=True)
    doc.units = ezdxf.units.MM
    msp = doc.modelspace()
    # A - L-bracket: LINE + ARC outline (must be chained), two D12 holes
    ox, oy, r = 0.0, 0.0, 20.0
    pts = [(ox, oy), (ox + 240, oy), (ox + 240, oy + 50), (ox + 50 + r, oy + 50)]
    for a, b in zip(pts, pts[1:]):
        msp.add_line(a, b)
    msp.add_arc((ox + 50 + r, oy + 50 + r), r, 180, 270)            # inner fillet
    msp.add_line((ox + 50, oy + 50 + r), (ox + 50, oy + 170))
    msp.add_line((ox + 50, oy + 170), (ox, oy + 170))
    msp.add_line((ox, oy + 170), (ox, oy))
    msp.add_circle((ox + 200, oy + 25), 6)
    msp.add_circle((ox + 25, oy + 140), 6)
    # B - plate with rounded corners (bulged LWPOLYLINE), 4 holes and a slot
    ox, oy, w, h, rr = 400.0, 0.0, 300.0, 180.0, 15.0
    b = math.tan(math.radians(90) / 4)
    msp.add_lwpolyline([(ox + rr, oy, 0), (ox + w - rr, oy, b), (ox + w, oy + rr, 0), (ox + w, oy + h - rr, b),
                        (ox + w - rr, oy + h, 0), (ox + rr, oy + h, b), (ox, oy + h - rr, 0), (ox, oy + rr, b)],
                       format="xyb", close=True)
    for hx, hy in ((30, 30), (w - 30, 30), (w - 30, h - 30), (30, h - 30)):
        msp.add_circle((ox + hx, oy + hy), 5)
    msp.add_lwpolyline([(ox + 110, oy + 80, 0), (ox + 190, oy + 80, 1), (ox + 190, oy + 100, 0),
                        (ox + 110, oy + 100, 1)], format="xyb", close=True)
    # C - flange: D200 disc, D80 bore, 6 x D14 bolt holes
    cx, cy = 900.0, 100.0
    msp.add_circle((cx, cy), 100)
    msp.add_circle((cx, cy), 40)
    for k in range(6):
        a = math.radians(60 * k)
        msp.add_circle((cx + 70 * math.cos(a), cy + 70 * math.sin(a)), 7)
    # D - gusset: chamfered right triangle, one hole
    ox, oy = 1100.0, 0.0
    msp.add_lwpolyline([(ox, oy), (ox + 200, oy), (ox + 200, oy + 20), (ox + 20, oy + 150), (ox, oy + 150)], close=True)
    msp.add_circle((ox + 40, oy + 30), 8)
    doc.saveas(path)
    return path

In [ ]:
%%writefile smartnest_nesting.py
"""
Bansali SmartNest - irregular nesting into the MEASURED material (sheet minus existing cut-outs).

Geometry kernel (exact polygon maths, no bounding-box approximations):
    convex_decompose(P)         constrained Delaunay triangles merged into convex pieces
    calculate_nfp(A, B)         No-Fit Polygon A (+) (-B): positions of B's reference point at
                                which B overlaps A. Union of convex hulls of piece-pair sums.
    inner_fit(C, S)             positions of S's reference point at which S lies inside the
                                container C (which may have holes): (C - b0) minus the sweep of
                                -S along every edge of C.
Placement:
    feasible = IFP(part, rot) - union(NFP(placed_i, part)) ; a linear "bottom-left" score is
    optimal at a vertex of that region, so its vertices are the candidates. The chosen position
    is then VERIFIED with exact Shapely containment and distance checks (kernel proposes,
    polygon geometry decides). NFP/IFP use slightly conservative shapes, so rounding can only
    lose a placement, never create an overlap.
Optimisation:
    greedy decoder over a part sequence (best rotation per part), evolutionary search over
    sequences until max_time_s or no improvement; the best VALID layout is always returned.
Coordinates: machine/CAD millimetres, Y up, origin = bed bottom-left (the DXF frame).
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Sequence

import numpy as np
import shapely
from shapely import affinity
from shapely.geometry import MultiPolygon, Polygon, box
from shapely.geometry.polygon import orient
from shapely.ops import unary_union
from shapely.strtree import STRtree
from shapely.validation import make_valid

EPS_MM = 0.01            # extra spacing inside the kernel so exact checks pass with margin


@dataclass
class NestConfig:
    kerf_mm: float = 0.3                 # laser kerf width
    clearance_mm: float = 3.0            # minimum material web between neighbouring parts' kerfs
    edge_margin_mm: float = 5.0          # minimum web between a part's kerf and a material edge
    measurement_uncertainty_mm: float = 3.0   # vision uncertainty; see smartnest_vision.usable_region
    rotation_step_deg: float = 90.0      # allowed rotations 0, step, 2*step ... unless a part overrides
    max_time_s: float = 30.0             # MAX_OPTIMIZATION_TIME_SECONDS
    seed: int = 0
    geometry_tol_mm: float = 0.25        # conservative simplification used inside the kernel only
    min_remnant_mm: float = 100.0        # leftover narrower than this is counted as scrap
    gravity_y_weight: float = 0.25       # bottom-left score = right edge + w * top edge
    population: int = 8
    stall_generations: int = 40
    w_unplaced: float = 1e6              # objective weights (lower score is better)
    w_unplaced_area: float = 1e3
    w_envelope: float = 100.0
    w_cut_per_m: float = 1e-3

    @property
    def part_gap_mm(self) -> float:      # required distance between two real part edges
        return self.kerf_mm + self.clearance_mm

    @property
    def edge_gap_mm(self) -> float:      # required distance from a real part edge to material edge
        return self.kerf_mm / 2 + self.edge_margin_mm + self.measurement_uncertainty_mm


# ---------------------------------------------------------------------------
# Geometry kernel
# ---------------------------------------------------------------------------
def _as_area(g):
    parts = [p for p in getattr(g, "geoms", [g]) if isinstance(p, Polygon) and not p.is_empty] if g is not None else []
    if not parts and hasattr(g, "geoms"):
        parts = [q for p in g.geoms for q in getattr(p, "geoms", [p]) if isinstance(q, Polygon) and not q.is_empty]
    if not parts:
        return Polygon()
    return parts[0] if len(parts) == 1 else MultiPolygon(parts)


def grow_simplify(g, tol):
    """Fewer vertices, never smaller than g (buffer out by tol, then simplify by tol)."""
    return _as_area(make_valid(g.buffer(tol, join_style="mitre", mitre_limit=2.0).simplify(tol, preserve_topology=True)))


def shrink_simplify(g, tol):
    """Fewer vertices, never larger than g."""
    return _as_area(make_valid(g.buffer(-tol, join_style="mitre", mitre_limit=2.0).simplify(tol, preserve_topology=True)))


def convex_decompose(poly: Polygon) -> list[np.ndarray]:
    """Convex pieces covering the polygon's outline (holes are filled: parts are not nested
    inside other parts' holes in this version)."""
    solid = orient(Polygon(poly.exterior))
    hull = solid.convex_hull
    if solid.area >= hull.area * (1 - 1e-9):
        return [np.asarray(hull.exterior.coords)[:-1]]
    pieces = [orient(t) for t in shapely.constrained_delaunay_triangles(solid).geoms if t.area > 1e-12]
    merged = True
    while merged and len(pieces) > 1:            # greedy Hertel-Mehlhorn style merge
        merged = False
        tree = STRtree(pieces)
        for i, p in enumerate(pieces):
            for j in tree.query(p):
                if j <= i or p.intersection(pieces[j]).length < 1e-9:
                    continue
                u = p.union(pieces[j])
                if isinstance(u, Polygon) and u.area >= u.convex_hull.area * (1 - 1e-9):
                    pieces[i] = orient(u.convex_hull)
                    del pieces[j]
                    merged = True
                    break
            if merged:
                break
    return [np.asarray(p.exterior.coords)[:-1] for p in pieces]


def _pad(pieces: Sequence[np.ndarray]) -> np.ndarray:
    k = max(len(p) for p in pieces)
    return np.stack([np.vstack([p, np.repeat(p[-1:], k - len(p), axis=0)]) for p in pieces])


def _hull_union(point_sets: np.ndarray):
    n, k, _ = point_sets.shape
    mp = shapely.multipoints(point_sets.reshape(-1, 2), indices=np.repeat(np.arange(n), k))
    return shapely.union_all(shapely.convex_hull(mp))


def minkowski_sum(a_pieces: Sequence[np.ndarray], b_pieces: Sequence[np.ndarray]):
    A, B = _pad(a_pieces), _pad(b_pieces)
    s = A[:, None, :, None, :] + B[None, :, None, :, :]
    return _as_area(_hull_union(s.reshape(len(A) * len(B), -1, 2)))


def calculate_nfp(a_pieces, b_pieces):
    """No-Fit Polygon of B orbiting A (both with reference point at their own origin)."""
    return minkowski_sum(a_pieces, [-p for p in b_pieces])


def ring_edges(g) -> np.ndarray:
    segs = []
    for p in getattr(g, "geoms", [g]):
        for r in [p.exterior, *p.interiors]:
            c = np.asarray(r.coords)
            segs.append(np.stack([c[:-1], c[1:]], axis=1))
    return np.concatenate(segs) if segs else np.zeros((0, 2, 2))


def inner_fit(container, edges: np.ndarray, s_pieces: Sequence[np.ndarray], b0: tuple[float, float],
              chunk: int = 4000):
    """Positions t with S + t inside the container (exact for any container, holes included):
    b0 + t must be inside, and S + t must not touch any container edge."""
    neg = _pad([-p for p in s_pieces])
    forb = []
    for i in range(0, len(edges), chunk):
        e = edges[i:i + chunk]
        pts = e[:, None, :, None, :] + neg[None, :, None, :, :]          # (E, P, 2, k, 2)
        forb.append(_hull_union(pts.reshape(len(e) * len(neg), -1, 2)))
    region = affinity.translate(container, -b0[0], -b0[1])
    return _as_area(region.difference(shapely.union_all(forb)))


# ---------------------------------------------------------------------------
# Results
# ---------------------------------------------------------------------------
@dataclass
class Placement:
    part_index: int
    part_name: str
    instance_id: str
    x_mm: float                     # position of the part's centroid (CAD mm)
    y_mm: float
    rotation_deg: float             # counter-clockwise, about the centroid
    geometry: Polygon               # real part outline incl. holes, CAD mm


@dataclass
class NestResult:
    placements: list[Placement]
    required: int
    unplaced: dict[str, int]
    metrics: dict[str, Any]
    checks: dict[str, tuple[bool, str]]
    notes: list[str] = field(default_factory=list)
    history: list[tuple[float, float]] = field(default_factory=list)
    evaluations: int = 0
    time_s: float = 0.0
    config: NestConfig | None = None

    @property
    def placed(self) -> int:
        return len(self.placements)

    @property
    def export_enabled(self) -> bool:
        return all(ok for ok, _ in self.checks.values())

    def failed_checks(self) -> list[str]:
        return [f"{k}: {m}" for k, (ok, m) in self.checks.items() if not ok]

    @property
    def message(self) -> str:
        if self.placed == self.required:
            return f"All {self.required} parts fit within the available material."
        return f"{self.placed} of {self.required} parts fit within the available material."

    def to_dict(self) -> dict[str, Any]:
        m = self.metrics
        return {
            "required_parts": self.required, "placed_parts": self.placed,
            "unplaced_parts": self.required - self.placed, "unplaced_by_part": self.unplaced,
            "message": self.message,
            "utilization_percent": round(m["utilization_percent"], 2),
            "waste_percent": round(m["waste_percent"], 2),
            "waste_area_mm2": round(m["waste_area_mm2"], 0),
            "reusable_remnant_mm2": round(m["reusable_remnant_mm2"], 0),
            "scrap_area_mm2": round(m["scrap_area_mm2"], 0),
            "cutting_distance_mm": round(m["cutting_distance_mm"], 1),
            "pierces": m["pierces"], "travel_distance_mm": round(m["travel_distance_mm"], 1),
            "placements": [{"part_id": p.instance_id, "part": p.part_name, "x_mm": round(p.x_mm, 3),
                            "y_mm": round(p.y_mm, 3), "rotation_deg": p.rotation_deg} for p in self.placements],
            "checks": {k: {"ok": ok, "detail": msg} for k, (ok, msg) in self.checks.items()},
            "export_enabled": self.export_enabled, "notes": self.notes,
            "evaluations": self.evaluations, "time_s": round(self.time_s, 2),
        }


# ---------------------------------------------------------------------------
# Kernel with caches
# ---------------------------------------------------------------------------
@dataclass
class _Shape:
    real: Polygon
    ifp_pieces: list[np.ndarray]
    ifp_ref: tuple[float, float]
    eff_pieces: list[np.ndarray]
    bounds: tuple[float, float, float, float]


class _Kernel:
    def __init__(self, parts, container, cfg: NestConfig):
        self.parts, self.cfg = parts, cfg
        self.container = container
        shapely.prepare(self.container)
        self.cc = shrink_simplify(container, cfg.geometry_tol_mm)
        self.edges = ring_edges(self.cc)
        self._shape, self._ifp, self._nfp, self._rot = {}, {}, {}, {}

    def rotations(self, pi: int) -> list[float]:
        if pi in self._rot:
            return self._rot[pi]
        p = self.parts[pi]
        cand = p.allowed_rotations if p.allowed_rotations is not None else \
            list(np.arange(0.0, 360.0, self.cfg.rotation_step_deg))
        keep, shapes = [], []
        base = Polygon(p.polygon.exterior)
        for r in cand:
            g = affinity.rotate(base, float(r), origin=(0, 0))
            if all(g.symmetric_difference(s).area > 1e-4 * base.area for s in shapes):   # skip symmetric repeats
                keep.append(float(r) % 360.0)
                shapes.append(g)
        self._rot[pi] = keep
        return keep

    def shape(self, pi: int, rot: float) -> _Shape:
        key = (pi, rot)
        if key not in self._shape:
            c, tol = self.cfg, self.cfg.geometry_tol_mm
            real = orient(affinity.rotate(self.parts[pi].polygon, rot, origin=(0, 0)))
            solid = Polygon(real.exterior)
            s_c = grow_simplify(solid, tol)
            eff = solid.buffer(c.part_gap_mm / 2 + EPS_MM, join_style="mitre", mitre_limit=2.0)
            e_c = grow_simplify(eff, tol)
            b0 = s_c.representative_point()
            self._shape[key] = _Shape(real, convex_decompose(s_c), (b0.x, b0.y), convex_decompose(e_c), real.bounds)
        return self._shape[key]

    def ifp(self, pi: int, rot: float):
        key = (pi, rot)
        if key not in self._ifp:
            sh = self.shape(pi, rot)
            g = inner_fit(self.cc, self.edges, sh.ifp_pieces, sh.ifp_ref) if not self.cc.is_empty else Polygon()
            self._ifp[key] = (g, g.bounds if not g.is_empty else None)
        return self._ifp[key]

    def nfp(self, pa, ra, pb, rb):
        key = (pa, ra, pb, rb)
        if key not in self._nfp:
            g = calculate_nfp(self.shape(pa, ra).eff_pieces, self.shape(pb, rb).eff_pieces)
            self._nfp[key] = (g, g.bounds)
        return self._nfp[key]

    def exact_ok(self, real_t: Polygon, placed) -> bool:
        if not self.container.contains(real_t):
            return False
        gap = self.cfg.part_gap_mm - 1e-6
        x0, y0, x1, y1 = real_t.bounds
        for q in placed:
            qb = q[5]
            if qb[0] - gap > x1 or qb[2] + gap < x0 or qb[1] - gap > y1 or qb[3] + gap < y0:
                continue
            if real_t.distance(q[4]) < gap:
                return False
        return True

    def decode(self, seq: Sequence[int], deadline: float | None = None):
        """Greedy bottom-left construction. Returns (placed, unplaced) or None if out of time.

        Each (part type, rotation) keeps its own feasible region and subtracts only the NFPs of
        parts placed since it was last used; a part type whose regions are all empty is skipped
        for the rest of the sequence (regions only ever shrink)."""
        gy = self.cfg.gravity_y_weight
        placed, unplaced = [], []
        free: dict[tuple[int, float], list] = {}
        dead: set[int] = set()
        for pi in seq:
            if deadline is not None and time.perf_counter() > deadline:
                return None
            if pi in dead:
                unplaced.append(pi)
                continue
            best, any_region = None, False
            for rot in self.rotations(pi):
                key = (pi, rot)
                if key not in free:
                    ifp, ib = self.ifp(pi, rot)
                    free[key] = [ifp, 0]
                region, n = free[key]
                if region.is_empty:
                    continue
                if n < len(placed):
                    rb = region.bounds
                    obs = []
                    for q in placed[n:]:
                        g, nb = self.nfp(q[0], q[1], pi, rot)
                        if nb[0] + q[2] > rb[2] or nb[2] + q[2] < rb[0] or nb[1] + q[3] > rb[3] or nb[3] + q[3] < rb[1]:
                            continue
                        obs.append(affinity.translate(g, q[2], q[3]))
                    if obs:
                        region = _as_area(region.difference(shapely.union_all(obs)))
                    free[key] = [region, len(placed)]
                if region.is_empty:
                    continue
                any_region = True
                sh = self.shape(pi, rot)
                V = shapely.get_coordinates(region)
                score = V[:, 0] + sh.bounds[2] + gy * (V[:, 1] + sh.bounds[3])
                for k in np.argsort(score)[:12]:
                    if best is not None and score[k] >= best[0]:
                        break
                    real_t = affinity.translate(sh.real, V[k, 0], V[k, 1])
                    if self.exact_ok(real_t, placed):
                        best = (float(score[k]), rot, float(V[k, 0]), float(V[k, 1]), real_t)
                        break
            if best is None:
                unplaced.append(pi)
                if not any_region:
                    dead.add(pi)
            else:
                _, rot, x, y, real_t = best
                placed.append((pi, rot, x, y, real_t, real_t.bounds))
        return placed, unplaced


# ---------------------------------------------------------------------------
# Optimiser
# ---------------------------------------------------------------------------
def material_to_cad(material_bed, bed_h_mm: float):
    """Vision frame (y down) -> machine frame (Y up, origin bottom-left)."""
    g = affinity.translate(affinity.scale(material_bed, 1.0, -1.0, origin=(0, 0)), 0.0, bed_h_mm)
    return _as_area(make_valid(orient(g) if isinstance(g, Polygon) else g))


def cad_to_bed(geom, bed_h_mm: float):
    return affinity.translate(affinity.scale(geom, 1.0, -1.0, origin=(0, 0)), 0.0, bed_h_mm)


def _score(kernel: _Kernel, placed, unplaced, required_area) -> float:
    c = kernel.cfg
    un_area = sum(kernel.parts[pi].area_mm2 for pi in unplaced)
    if placed:
        b = np.array([q[5] for q in placed])
        cb = kernel.container.bounds
        env = (b[:, 2].max() - cb[0]) * (b[:, 3].max() - cb[1])
        cut = sum(kernel.parts[q[0]].cut_length_mm for q in placed)
    else:
        env, cut = 0.0, 0.0
    return (c.w_unplaced * len(unplaced) + c.w_unplaced_area * un_area / max(required_area, 1.0)
            + c.w_envelope * env / max(kernel.container.area, 1.0) + c.w_cut_per_m * cut / 1000.0)


def _mutate(seq: list[int], rng: np.random.Generator) -> list[int]:
    s = list(seq)
    n = len(s)
    if n < 2:
        return s
    move = rng.integers(3)
    i, j = sorted(rng.choice(n, 2, replace=False))
    if move == 0:
        s[i], s[j] = s[j], s[i]
    elif move == 1:
        s.insert(j, s.pop(i))
    else:
        s[i:j + 1] = s[i:j + 1][::-1]
    return s


def nest(parts, material_cad, bed_w_mm: float, bed_h_mm: float, cfg: NestConfig | None = None,
         progress: Callable[[str, dict], None] | None = None, scan_problems: Sequence[str] = ()) -> NestResult:
    """Place every requested part instance into the material, maximising parts placed, then
    compactness (large reusable remnant), then cutting distance."""
    cfg = cfg or NestConfig()
    t0 = time.perf_counter()
    deadline = t0 + cfg.max_time_s
    say = progress or (lambda stage, info: None)
    rng = np.random.default_rng(cfg.seed)
    notes: list[str] = []

    say("Detecting geometry", {})
    material_cad = _as_area(make_valid(material_cad))
    container = _as_area(material_cad.buffer(-cfg.edge_gap_mm, join_style="mitre", mitre_limit=5.0))
    kernel = _Kernel(parts, container, cfg)
    instances = [pi for pi, p in enumerate(parts) for _ in range(p.quantity)]
    required_area = sum(parts[pi].area_mm2 for pi in instances)

    say("Generating candidate placements", {"part_types": len(parts)})
    for pi, p in enumerate(parts):
        if all(kernel.ifp(pi, r)[1] is None for r in kernel.rotations(pi)):
            notes.append(f"Part {p.name} is larger than the remaining usable region "
                         f"(after {cfg.edge_gap_mm:.1f} mm edge allowance) at every allowed rotation.")
    fits = [pi for pi in instances if any(kernel.ifp(pi, r)[1] is not None for r in kernel.rotations(pi))]
    never = [pi for pi in instances if pi not in set(fits)]

    say("Optimizing nesting", {"instances": len(instances)})
    hull_gap = {pi: 1 - p.area_mm2 / Polygon(p.polygon.exterior).convex_hull.area for pi, p in enumerate(parts)}
    diag = {pi: math.hypot(p.polygon.bounds[2] - p.polygon.bounds[0], p.polygon.bounds[3] - p.polygon.bounds[1])
            for pi, p in enumerate(parts)}
    seeds = [sorted(fits, key=lambda pi: -parts[pi].area_mm2),
             sorted(fits, key=lambda pi: -diag[pi]),
             sorted(fits, key=lambda pi: (-round(hull_gap[pi], 1), -parts[pi].area_mm2)),
             sorted(fits, key=lambda pi: parts[pi].area_mm2)]        # count is the primary objective
    pop, seen, history, evals = [], set(), [], 0
    best = None
    for s in seeds:
        if tuple(s) in seen:
            continue
        seen.add(tuple(s))
        res = kernel.decode(s, deadline if pop else None)          # the first layout always completes
        if res is None:
            break
        evals += 1
        sc = _score(kernel, res[0], res[1] + never, required_area)
        pop.append((sc, s, res))
        if best is None or sc < best[0]:
            best = (sc, s, res)
            history.append((time.perf_counter() - t0, sc))
    stall = 0
    while time.perf_counter() < deadline and stall < cfg.stall_generations and len(fits) > 1:
        k = rng.choice(len(pop), size=min(2, len(pop)), replace=False)
        parent = min((pop[i] for i in k), key=lambda t: t[0])
        child = _mutate(parent[1], rng)
        for _ in range(5):
            if tuple(child) not in seen:
                break
            child = _mutate(child, rng)
        if tuple(child) in seen:
            stall += 1
            continue
        seen.add(tuple(child))
        res = kernel.decode(child, deadline)
        if res is None:
            break
        evals += 1
        sc = _score(kernel, res[0], res[1] + never, required_area)
        if sc < best[0] - 1e-9:
            best, stall = (sc, child, res), 0
            history.append((time.perf_counter() - t0, sc))
            say("Optimizing nesting", {"evaluations": evals, "best_score": sc,
                                       "placed": len(res[0]), "required": len(instances)})
        else:
            stall += 1
        if len(pop) < cfg.population:
            pop.append((sc, child, res))
        else:
            worst = max(range(len(pop)), key=lambda i: pop[i][0])
            if sc < pop[worst][0]:
                pop[worst] = (sc, child, res)
    if time.perf_counter() >= deadline:
        notes.append(f"Optimization time limit ({cfg.max_time_s:.0f} s) reached; best layout found is returned.")

    say("Generating final layout", {})
    placed, unplaced = best[2]
    unplaced = unplaced + never
    counters: dict[int, int] = {}
    placements = []
    for pi, rot, x, y, real_t, _ in sorted(placed, key=lambda q: (parts[q[0]].name, q[2], q[3])):
        counters[pi] = counters.get(pi, 0) + 1
        placements.append(Placement(pi, parts[pi].name, f"{parts[pi].name}_{counters[pi]:02d}", x, y, rot, real_t))
    unplaced_by = {}
    for pi in unplaced:
        unplaced_by[parts[pi].name] = unplaced_by.get(parts[pi].name, 0) + 1
    if unplaced and not notes:
        notes.append("No feasible arrangement was found for all requested parts.")
    metrics = layout_metrics(placements, parts, material_cad, cfg)
    say("Validating layout", {})
    checks = validate_layout(placements, parts, material_cad, container, bed_w_mm, bed_h_mm, cfg, scan_problems)
    return NestResult(placements, len(instances), unplaced_by, metrics, checks, notes, history, evals,
                      time.perf_counter() - t0, cfg)


# ---------------------------------------------------------------------------
# Metrics and validation
# ---------------------------------------------------------------------------
def layout_metrics(placements, parts, material_cad, cfg: NestConfig) -> dict[str, Any]:
    available = material_cad.area
    part_area = sum(p.geometry.area for p in placements)
    cut = sum(p.geometry.exterior.length + sum(r.length for r in p.geometry.interiors) for p in placements)
    pierces = sum(1 + len(p.geometry.interiors) for p in placements)
    used = unary_union([p.geometry.buffer(cfg.kerf_mm / 2 + cfg.clearance_mm, join_style="mitre")
                        for p in placements]) if placements else Polygon()
    remaining = _as_area(material_cad.difference(used))
    r = cfg.min_remnant_mm / 2
    reusable = _as_area(remaining.buffer(-r, join_style="mitre").buffer(r, join_style="mitre").intersection(remaining))
    waste = available - part_area
    # rapid travel: nearest-neighbour tour from the machine origin through the part centroids
    pts = [np.array([p.x_mm, p.y_mm]) for p in placements]
    cur, travel = np.zeros(2), 0.0
    while pts:
        k = int(np.argmin([np.hypot(*(q - cur)) for q in pts]))
        travel += float(np.hypot(*(pts[k] - cur)))
        cur = pts.pop(k)
    return {"available_area_mm2": available, "part_area_mm2": part_area,
            "utilization_percent": 100 * part_area / available if available else 0.0,
            "waste_area_mm2": waste, "waste_percent": 100 * waste / available if available else 0.0,
            "reusable_remnant_mm2": reusable.area, "scrap_area_mm2": max(0.0, waste - reusable.area),
            "kerf_loss_mm2": cut * cfg.kerf_mm, "cutting_distance_mm": cut, "pierces": pierces,
            "travel_distance_mm": travel, "reusable_remnant": reusable}


def validate_layout(placements, parts, material_cad, container, bed_w, bed_h, cfg: NestConfig,
                    scan_problems: Sequence[str] = ()) -> dict[str, tuple[bool, str]]:
    """The export gate: every check must pass."""
    checks: dict[str, tuple[bool, str]] = {}
    checks["sheet"] = (bool(material_cad.is_valid and material_cad.area > 0),
                       "material geometry valid" if material_cad.is_valid else "material geometry invalid")
    checks["cutouts"] = (not scan_problems, "; ".join(scan_problems) or "scan geometry consistent")
    bad_parts = [p.name for p in parts if not (p.polygon.is_valid and p.polygon.area > 0)]
    checks["parts"] = (not bad_parts, f"invalid parts: {bad_parts}" if bad_parts else f"{len(parts)} part types valid")
    tolv = 1e-6
    outside = [p.instance_id for p in placements if p.geometry.difference(material_cad).area > tolv]
    checks["inside_sheet"] = (not outside, f"outside material: {outside}" if outside else "all parts on material")
    edge = [p.instance_id for p in placements if not container.buffer(tolv).contains(p.geometry)]
    checks["placements"] = (not edge, f"closer than {cfg.edge_gap_mm:.2f} mm to a material edge: {edge}" if edge
                            else f"all parts >= {cfg.edge_gap_mm:.2f} mm from sheet edges and existing cut-outs")
    geoms = [p.geometry for p in placements]
    tree = STRtree(geoms) if geoms else None
    overlaps, close = [], []
    for i, g in enumerate(geoms):
        for j in tree.query(g.buffer(cfg.part_gap_mm)):
            if j <= i:
                continue
            if g.intersection(geoms[j]).area > tolv:
                overlaps.append((placements[i].instance_id, placements[j].instance_id))
            elif g.distance(geoms[j]) < cfg.part_gap_mm - 1e-6:
                close.append((placements[i].instance_id, placements[j].instance_id, round(g.distance(geoms[j]), 3)))
    checks["no_overlap"] = (not overlaps, f"overlapping: {overlaps}" if overlaps else "no overlapping parts")
    checks["clearance"] = (not close, f"below {cfg.part_gap_mm:.2f} mm: {close}" if close
                           else f"all part gaps >= kerf + clearance = {cfg.part_gap_mm:.2f} mm")
    bed = box(0, 0, bed_w, bed_h).buffer(tolv)
    oob = [p.instance_id for p in placements if not bed.contains(p.geometry)]
    checks["machine_bounds"] = (not oob, f"outside machine bed: {oob}" if oob else "inside machine bed")
    return checks


def export_cutting_file(nest_result: NestResult, parts, path: str, **kw) -> dict[str, Any]:
    """Write the DXF, read it back, compare. The file is kept only if the comparison passes."""
    import os

    from smartnest_cad import export_layout_dxf, validate_layout_dxf

    tmp = path + ".tmp.dxf"
    info = export_layout_dxf(nest_result, parts, tmp, **kw)
    check = validate_layout_dxf(tmp, nest_result)
    nest_result.checks["dxf_geometry"] = (check["ok"], f"re-read {check['parts_read']} parts, "
                                          f"difference {check['symmetric_difference_mm2']:.2f} mm2")
    if not check["ok"]:
        os.remove(tmp)
        raise ValueError(f"Export failed validation: {check}")
    os.replace(tmp, path)
    info["path"] = path
    info["validation"] = check
    return info


def draw_layout(ortho_bgr: np.ndarray, rect, nest_result: NestResult, bed_h_mm: float) -> np.ndarray:
    """New parts on the measured bed image: blue fill, white outline, part id."""
    import cv2

    vis = ortho_bgr.copy()
    over = vis.copy()
    SH = 4
    for p in nest_result.placements:
        g = cad_to_bed(p.geometry, bed_h_mm)
        rings = [g.exterior, *g.interiors]
        pts = [np.round(rect.mm_to_px(np.asarray(r.coords)) * (1 << SH)).astype(np.int32).reshape(-1, 1, 2) for r in rings]
        cv2.fillPoly(over, pts[:1], (200, 110, 20), cv2.LINE_AA, SH)
        for q in pts[1:]:
            cv2.fillPoly(over, [q], (30, 30, 30), cv2.LINE_AA, SH)
    cv2.addWeighted(over, 0.7, vis, 0.3, 0, vis)
    fs = max(0.35, min(vis.shape[:2]) / 1800)
    for p in nest_result.placements:
        g = cad_to_bed(p.geometry, bed_h_mm)
        for r in [g.exterior, *g.interiors]:
            q = np.round(rect.mm_to_px(np.asarray(r.coords)) * (1 << SH)).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(vis, [q], True, (255, 255, 255), 1, cv2.LINE_AA, SH)
        cx, cy = rect.mm_to_px(np.asarray(cad_to_bed(shapely.Point(p.x_mm, p.y_mm), bed_h_mm).coords))[0]
        cv2.putText(vis, p.instance_id, (int(cx - 25 * fs), int(cy + 5 * fs)), cv2.FONT_HERSHEY_SIMPLEX, fs,
                    (255, 255, 255), 1, cv2.LINE_AA)
    return vis

In [ ]:
import json, math, base64, os
import cv2, numpy as np, matplotlib.pyplot as plt
import smartnest_vision as sv, smartnest_synthetic as ss, smartnest_cad as cad, smartnest_nesting as sn
try:
    from google.colab.output import eval_js
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def show(img, title='', w=15):
    plt.figure(figsize=(w, w * img.shape[0] / img.shape[1]))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis('off'); plt.show()
print('SmartNest modules loaded. OpenCV', cv2.__version__)

## Click 1 - SCAN BED

In [ ]:
#@title Acquire the bed image
CAPTURE_MODE = 'demo'  # @param ['webcam', 'upload', 'demo']
DEMO_SCENE = 'mixed_holes'  # @param ['fresh_sheet', 'four_circles', 'mixed_holes', 'irregular_remnant', 'dark_steel', 'lens_distortion', 'small_bed']
BED_WIDTH_MM = 1500   # @param {type:"number"}
BED_HEIGHT_MM = 950   # @param {type:"number"}
CAMERA_NAME = ''  # @param {type:"string"}
# Leave CAMERA_NAME empty: the external webcam is chosen automatically (and remembered).
# Only type part of a name (e.g. 'LAPCARE') if the automatic choice is ever wrong.

import os, base64, cv2, numpy as np
try:
    from google.colab.output import eval_js
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
try:
    import smartnest_vision as sv, smartnest_synthetic as ss
except ImportError:
    raise SystemExit('The SmartNest modules are not in this runtime yet. Use Runtime -> Run all '
                     '(or run every cell above this one once), then run this cell again.')
if 'show' not in globals():
    import matplotlib.pyplot as plt
    def show(img, title='', w=15):
        plt.figure(figsize=(w, w * img.shape[0] / img.shape[1]))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis('off'); plt.show()

CAMERA_MEMORY = 'smartnest_camera.txt'

def capture_from_browser_webcam(camera_name='', quality=0.92):
    # Browsers open the DEFAULT camera (the laptop's own) unless a device is chosen explicitly.
    # Choice order: CAMERA_NAME > camera used last time > known external webcam brand >
    # the most recently connected camera that does not look built-in.
    if not camera_name and os.path.exists(CAMERA_MEMORY):
        camera_name = open(CAMERA_MEMORY).read().split(' (')[0]
    want = ''.join(ch for ch in camera_name if ch.isalnum() or ch in ' -_.')
    js = '''
    (async () => {
      const WANT = 'CAMNAME'.toLowerCase().trim();
      const builtIn = /integrated|built-?in|internal|facetime|front|truevision|easycamera|hd webcam|uvc webcam|ir camera|infrared|hello|virtual/i;
      const external = /lapcare|logitech|c9[0-9][0-9]|c270|c310|c505|c615|brio|lifecam|razer|elgato|zebronics|frontech|iball|a4tech|quantum|creative|usb camera|usb video|external/i;
      const box = document.createElement('div');
      box.style.cssText = 'background:#0f172a;padding:16px;border-radius:10px;border:2px solid #00e5ff;max-width:760px;color:#f8fafc;font-family:monospace';
      box.innerHTML = '<h3 style="margin:0 0 8px;color:#00e5ff">LIVE OVERHEAD CAMERA</h3>' +
        '<p style="margin:0 0 10px;color:#94a3b8">Check that all 4 bed corners (or the markers) are visible, then capture. Wrong camera? Pick another below.</p>';
      const sel = document.createElement('select');
      sel.style.cssText = 'width:100%;padding:8px;font-size:14px;margin-bottom:8px;background:#1e293b;color:#f8fafc;border:1px solid #00e5ff;border-radius:6px';
      const video = document.createElement('video'); video.style.width = '100%'; video.setAttribute('playsinline', ''); video.muted = true;
      const info = document.createElement('div'); info.style.cssText = 'color:#f59e0b;margin:6px 0';
      const btn = document.createElement('button'); btn.textContent = 'CAPTURE BED IMAGE';
      btn.style.cssText = 'margin-top:6px;background:#00e5ff;border:0;padding:10px 20px;font-weight:bold;border-radius:6px;cursor:pointer';
      box.append(sel, video, info, btn); document.body.appendChild(box);
      const videoInputs = async () => (await navigator.mediaDevices.enumerateDevices()).filter(d => d.kind === 'videoinput');
      let cams = await videoInputs();
      if (!cams.some(c => c.label)) {          // names are hidden until camera permission is granted (first run only)
        info.textContent = 'Allow camera access in the browser prompt...';
        const probe = await navigator.mediaDevices.getUserMedia({video: true});
        probe.getTracks().forEach(t => t.stop());
        cams = await videoInputs();
      }
      const fill = list => { sel.innerHTML = ''; list.forEach((c, i) => { const o = document.createElement('option');
        o.value = c.deviceId; o.text = c.label || ('Camera ' + (i + 1)); sel.appendChild(o); }); };
      fill(cams);
      // +2 known external webcam, -1 sounds built-in; ties go to the most recently connected camera
      const score = c => (external.test(c.label) ? 2 : 0) - (builtIn.test(c.label) ? 1 : 0);
      const best = list => list.reduce((a, c) => (score(c) >= score(a) ? c : a));
      let saved = null; try { saved = localStorage.getItem('smartnest_camera'); } catch (e) {}
      const pick = (WANT && cams.find(c => c.label.toLowerCase().includes(WANT)))
                || cams.find(c => c.deviceId === saved)
                || best(cams);
      sel.value = pick.deviceId;
      let stream = null;
      async function start(id) {
        btn.disabled = true; btn.style.opacity = 0.4; info.textContent = 'Opening camera...';
        if (stream) stream.getTracks().forEach(t => t.stop());
        stream = null;
        const s = await navigator.mediaDevices.getUserMedia({video: {deviceId: {exact: id}, width: {ideal: 1920}, height: {ideal: 1080}}});
        video.srcObject = s; await video.play(); stream = s;
        try { localStorage.setItem('smartnest_camera', id); } catch (e) {}
        const n = sel.options.length;
        info.textContent = 'Using: ' + sel.options[sel.selectedIndex].text + '  (' + video.videoWidth + ' x ' + video.videoHeight + ' px)' +
          (n === 1 ? '  - only ONE camera found: if the external webcam is plugged in, re-plug it (it switches automatically)' : '  - ' + n + ' cameras found');
        btn.disabled = false; btn.style.opacity = 1;
      }
      const fail = e => { info.textContent = 'Cannot open this camera (' + e.message + '). Close other apps using it (Camera, Teams, WhatsApp) or pick another.'; };
      sel.onchange = () => start(sel.value).catch(fail);
      navigator.mediaDevices.ondevicechange = async () => {  // webcam plugged in / unplugged while the preview is open
        const before = new Set([...sel.options].map(o => o.value));
        const curId = stream ? stream.getVideoTracks()[0].getSettings().deviceId : sel.value;
        const now = await videoInputs();
        if (!now.length) { info.textContent = 'No camera connected.'; return; }
        fill(now);
        const cur = now.find(c => c.deviceId === curId);
        const added = now.filter(c => !before.has(c.deviceId));
        const cand = added.length ? best(added) : null;
        if (cand && (!cur || score(cand) >= score(cur))) { sel.value = cand.deviceId; start(cand.deviceId).catch(fail); }
        else if (!cur) { sel.value = best(now).deviceId; start(sel.value).catch(fail); }
        else { sel.value = cur.deviceId; }
      };
      await start(sel.value).catch(fail);
      return new Promise(res => { btn.onclick = () => {
        if (!stream || !video.videoWidth) { info.textContent = 'No live picture yet - pick a working camera.'; return; }
        const c = document.createElement('canvas'); c.width = video.videoWidth; c.height = video.videoHeight;
        c.getContext('2d').drawImage(video, 0, 0);
        const label = sel.options[sel.selectedIndex].text;
        navigator.mediaDevices.ondevicechange = null;
        stream.getTracks().forEach(t => t.stop()); box.remove();
        res({image: c.toDataURL('image/jpeg', QUALITY), label: label, width: c.width, height: c.height}); }; });
    })()'''.replace('QUALITY', str(quality)).replace('CAMNAME', want)
    r = eval_js(js)
    open(CAMERA_MEMORY, 'w').write(r['label'])
    print(f"Captured from: {r['label']}  ({r['width']} x {r['height']} px)")
    return cv2.imdecode(np.frombuffer(base64.b64decode(r['image'].split(',')[1]), np.uint8), cv2.IMREAD_COLOR)

scene = None
if CAPTURE_MODE == 'webcam' and IN_COLAB:
    raw_image = capture_from_browser_webcam(CAMERA_NAME)
elif CAPTURE_MODE == 'upload' and IN_COLAB:
    up = files.upload(); raw_image = cv2.imread(list(up)[0])
else:
    if CAPTURE_MODE != 'demo':
        print(f"'{CAPTURE_MODE}' needs Google Colab - using the demo scene instead.")
    scene = ss.make_scene(DEMO_SCENE)
    raw_image = scene.image
    BED_WIDTH_MM, BED_HEIGHT_MM = scene.bed_w, scene.bed_h
    print('Demo scene:', scene.description)
if raw_image is None:
    raise SystemExit('No camera image.')
show(raw_image, f'Acquired image {raw_image.shape[1]} x {raw_image.shape[0]} px')

In [ ]:
#@title Calibrate pixels -> machine millimetres
CALIBRATION_MODE = 'aruco'  # @param ['click_corners', 'aruco', 'load_file', 'demo_truth']
CALIBRATION_FILE = 'calibration.json'  # @param {type:"string"}
USE_EMPTY_BED_REFERENCE = False  # @param {type:"boolean"}

def click_bed_corners(image):
    h, w = image.shape[:2]; dw = min(960, w); dh = int(h * dw / w)
    b64 = base64.b64encode(cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, 85])[1]).decode()
    js = f'''
    (async () => {{
      const box = document.createElement('div');
      box.style.cssText = 'background:#0f172a;padding:14px;border-radius:10px;border:2px solid #00e5ff;max-width:{dw + 30}px;color:#f8fafc;font-family:sans-serif';
      box.innerHTML = '<b style="color:#00e5ff">CLICK THE 4 BED CORNERS: TOP-LEFT, TOP-RIGHT, BOTTOM-RIGHT, BOTTOM-LEFT</b><div id="st" style="color:#f59e0b;margin:6px 0">Click 1 of 4</div>';
      const cv = document.createElement('canvas'); cv.width = {dw}; cv.height = {dh}; cv.style.cssText = 'width:100%;cursor:crosshair';
      const ok = document.createElement('button'); ok.textContent = 'CONFIRM CORNERS'; ok.style.display = 'none';
      const rs = document.createElement('button'); rs.textContent = 'Reset';
      box.appendChild(cv); box.appendChild(rs); box.appendChild(ok); document.body.appendChild(box);
      const ctx = cv.getContext('2d'); const img = new Image(); img.src = 'data:image/jpeg;base64,{b64}';
      await new Promise(r => img.onload = r); let pts = [];
      const draw = () => {{ ctx.drawImage(img, 0, 0, {dw}, {dh}); ctx.strokeStyle = '#00e5ff'; ctx.beginPath();
        pts.forEach((p, i) => i ? ctx.lineTo(p[0], p[1]) : ctx.moveTo(p[0], p[1])); if (pts.length == 4) ctx.closePath(); ctx.stroke();
        pts.forEach(p => {{ ctx.fillStyle = '#f59e0b'; ctx.beginPath(); ctx.arc(p[0], p[1], 6, 0, 7); ctx.fill(); }}); }};
      draw();
      cv.onclick = e => {{ if (pts.length >= 4) return; const r = cv.getBoundingClientRect();
        pts.push([(e.clientX - r.left) * cv.width / r.width, (e.clientY - r.top) * cv.height / r.height]); draw();
        document.getElementById('st').textContent = pts.length < 4 ? 'Click ' + (pts.length + 1) + ' of 4' : 'Done - confirm';
        if (pts.length == 4) ok.style.display = 'inline-block'; }};
      rs.onclick = () => {{ pts = []; ok.style.display = 'none'; draw(); }};
      return new Promise(res => ok.onclick = () => {{ box.remove(); res(pts.map(p => [p[0] * {w} / {dw}, p[1] * {h} / {dh}])); }});
    }})()'''
    return np.array(eval_js(js), dtype=np.float64)

size = (raw_image.shape[1], raw_image.shape[0])
K = D = None
if scene is not None and scene.hint.get('needs_lens_model'):
    K, D = scene.camera.K, scene.camera.dist        # demo: the lens model a chessboard calibration would give
if CALIBRATION_MODE == 'load_file' and os.path.exists(CALIBRATION_FILE):
    calib = sv.BedCalibration.load(CALIBRATION_FILE)
elif CALIBRATION_MODE == 'aruco' and (scene is None or scene.markers_mm):
    layout = scene.markers_mm if scene is not None else sv.aruco_layout(
        {0: (-110, 150), 1: (-110, BED_HEIGHT_MM - 150), 2: (BED_WIDTH_MM + 110, 150), 3: (BED_WIDTH_MM + 110, BED_HEIGHT_MM - 150)}, 90)
    calib = sv.calibrate_from_aruco(raw_image, layout, BED_WIDTH_MM, BED_HEIGHT_MM, camera_matrix=K, dist_coeffs=D)
elif CALIBRATION_MODE == 'click_corners' and IN_COLAB:
    clicks = click_bed_corners(raw_image)
    gray = cv2.cvtColor(raw_image, cv2.COLOR_BGR2GRAY)
    clicks, shift, snapped = sv.refine_corners(gray, clicks)
    calib = sv.calibrate_bed(clicks, BED_WIDTH_MM, BED_HEIGHT_MM, size, camera_matrix=K, dist_coeffs=D)
else:
    calib = sv.calibrate_bed(scene.corners_px, BED_WIDTH_MM, BED_HEIGHT_MM, size, camera_matrix=K, dist_coeffs=D, source='demo truth')
calib.save(CALIBRATION_FILE)
rect = sv.Rectifier(calib)
reference = scene.empty_bed if (scene is not None and (USE_EMPTY_BED_REFERENCE or scene.hint.get('needs_reference'))) else None
print(f"Calibration: {calib.source}, {calib.n_points} points, residual "
      + (f"{calib.rms_mm:.2f} mm" if calib.rms_mm is not None else "unknown (4 points)")
      + f", ortho scale {rect.s} mm/px")
for w in calib.warnings: print('  !', w)

## Click 2 - DETECT / CONFIRM MATERIAL

In [ ]:
#@title Detect sheet, existing cut-outs and usable material
MATERIAL_POLARITY = 'bright'  # @param ['bright', 'dark']
cfg_v = sv.VisionConfig(polarity=MATERIAL_POLARITY)
try:
    scan = sv.scan_sheet(raw_image, calib, cfg_v, reference_image=reference, rectifier=rect)
except sv.VisionError as e:
    raise SystemExit(f'SCAN FAILED: {e}')
show(sv.draw_blueprint(scan, rect), 'Measured material map (green = sheet, red = existing cut-outs, orange = edge notches)')
d = scan.to_dict(include_geometry=False)
print(f"Bed            {scan.bed_w_mm:.0f} x {scan.bed_h_mm:.0f} mm")
print(f"Sheet          {d['sheet']['length_mm']} x {d['sheet']['width_mm']} mm  (angle {d['sheet']['angle_deg']} deg)")
print(f"Sheet area     {scan.sheet_area_mm2 / 1e6:.4f} m2")
print(f"Cut-outs       {len(scan.cutouts)}  (removed {scan.removed_area_mm2 / 1e6:.4f} m2), edge notches {len(scan.edge_notches)}")
print(f"Available      {scan.available_area_mm2 / 1e6:.4f} m2")
print(f"Quality score  {scan.confidence:.2f}  -> {'CONFIRM / ADJUST' if scan.needs_confirmation else 'OK'}")
for w in scan.warnings: print('  !', w)
for c in scan.cutouts:
    extra = f"D{c['diameter_mm']:.1f}" if c['type'] == 'circle' else (f"{c['length_mm']:.1f} x {c['width_mm']:.1f}" if c['type'] == 'rectangle' else '')
    print(f"  {c['id']}  {c['type']:9s} at ({c['centroid_mm'][0]:.1f}, {c['centroid_mm'][1]:.1f}) mm  {extra}  area {c['area_mm2']:.0f} mm2")
if scene is not None:
    iou = scan.material.intersection(scene.material).area / scan.material.union(scene.material).area
    print(f"\nDEMO ground truth: usable area error {100 * (scan.available_area_mm2 / scene.material.area - 1):+.3f} %, "
          f"IoU {iou:.4f}, cut-outs {len(scan.cutouts)}/{len(scene.holes)}")

In [ ]:
#@title (Optional) Adjust: remove a false cut-out / add a missed one, then re-run this cell
REMOVE_CUTOUT_IDS = ''  # @param {type:"string"}
ADD_CIRCLE_CUTOUTS = ''  # @param {type:"string"}
# e.g. REMOVE_CUTOUT_IDS = 'cutout_003'   ADD_CIRCLE_CUTOUTS = '[[900, 500, 40]]'  (x_mm, y_mm, radius_mm)
if REMOVE_CUTOUT_IDS.strip() or ADD_CIRCLE_CUTOUTS.strip():
    adds = [{'center': [x, y], 'radius': r} for x, y, r in (json.loads(ADD_CIRCLE_CUTOUTS) if ADD_CIRCLE_CUTOUTS.strip() else [])]
    scan = sv.apply_corrections(scan, cfg_v, remove_ids=[s.strip() for s in REMOVE_CUTOUT_IDS.split(',') if s.strip()], add_cutouts_mm=adds)
    show(sv.draw_blueprint(scan, rect), 'Corrected material map')
    print(f"Available after correction: {scan.available_area_mm2 / 1e6:.4f} m2")
else:
    print('No corrections.')

## Click 3 - ADD JOB

In [ ]:
#@title Load the part DXF
JOB_SOURCE = 'sample'  # @param ['sample', 'upload']
QUANTITIES = '{"A": 12, "B": 10, "C": 8, "D": 14}'  # @param {type:"string"}
if JOB_SOURCE == 'upload' and IN_COLAB:
    up = files.upload(); name = list(up)[0]
    job_path = cad.safe_filename(name); open(job_path, 'wb').write(up[name])
else:
    job_path = cad.make_sample_job_dxf('sample_job.dxf')
parts, report = cad.read_dxf_parts(job_path)
for e in report.errors: print('ERROR:', e)
for w in report.warnings: print('warning:', w)
for p in parts:
    p.quantity = int(json.loads(QUANTITIES or '{}').get(p.name, p.quantity))
print(f"File {report.file} ({report.units}) - {len(parts)} part types, {sum(p.quantity for p in parts)} parts required")
for p in parts:
    s = p.summary(); print(f"  Part {s['name']}  x{p.quantity:<3d} {s['bbox_mm'][0]:.1f} x {s['bbox_mm'][1]:.1f} mm  area {s['area_mm2']:.0f} mm2  holes {s['holes']}")

## Click 4 - OPTIMIZE

In [ ]:
#@title Nest into the measured material
KERF_MM = 0.3  # @param {type:"number"}
CLEARANCE_MM = 3.0  # @param {type:"number"}
EDGE_MARGIN_MM = 5.0  # @param {type:"number"}
MEASUREMENT_UNCERTAINTY_MM = 3.0  # @param {type:"number"}
ROTATION_STEP_DEG = 90  # @param [15, 30, 45, 90]
MAX_OPTIMIZATION_TIME_SECONDS = 30  # @param {type:"number"}
cfg_n = sn.NestConfig(kerf_mm=KERF_MM, clearance_mm=CLEARANCE_MM, edge_margin_mm=EDGE_MARGIN_MM,
                      measurement_uncertainty_mm=MEASUREMENT_UNCERTAINTY_MM, rotation_step_deg=float(ROTATION_STEP_DEG),
                      max_time_s=float(MAX_OPTIMIZATION_TIME_SECONDS))
last = {'stage': None}
def progress(stage, info):
    if stage != last['stage']:
        print('->', stage); last['stage'] = stage
    elif 'placed' in info:
        print(f"   improved: {info['placed']}/{info['required']} placed after {info['evaluations']} layouts")
material_cad = sn.material_to_cad(scan.material, scan.bed_h_mm)
result = sn.nest(parts, material_cad, scan.bed_w_mm, scan.bed_h_mm, cfg_n, progress, scan_problems=sv.validate_scan(scan))
show(sn.draw_layout(sv.draw_blueprint(scan, rect, show_hud=False), rect, result, scan.bed_h_mm), 'Where the laser will cut (blue = new parts)')
r = result.to_dict(); m = result.metrics
print(result.message)
print(f"Parts placed      {r['placed_parts']} / {r['required_parts']}   {r['unplaced_by_part'] or ''}")
print(f"Utilization       {r['utilization_percent']:.1f} %   waste {r['waste_percent']:.1f} %")
print(f"Reusable remnant  {m['reusable_remnant_mm2'] / 1e6:.3f} m2   scrap {m['scrap_area_mm2'] / 1e6:.3f} m2")
print(f"Cutting distance  {m['cutting_distance_mm'] / 1000:.2f} m   pierces {m['pierces']}   rapid travel ~{m['travel_distance_mm'] / 1000:.1f} m")
print(f"Search            {result.evaluations} layouts in {result.time_s:.1f} s")
for n in result.notes: print('  note:', n)
print('Checks:'); [print(f"  {'PASS' if ok else 'FAIL'}  {k}: {msg}") for k, (ok, msg) in result.checks.items()]

## Click 5 - EXPORT CUTTING FILE

In [ ]:
#@title Export (only if every check passes)
if not result.export_enabled:
    print('EXPORT DISABLED:'); [print('  ', f) for f in result.failed_checks()]
else:
    info = sn.export_cutting_file(result, parts, 'smartnest_cut.dxf')
    sv.export_dxf(scan, 'smartnest_material_map.dxf')
    json.dump({'scan': scan.to_dict(), 'nest': result.to_dict()}, open('smartnest_job.json', 'w'), indent=1)
    cad.export_layout_svg(result, material_cad, (scan.bed_w_mm, scan.bed_h_mm), 'smartnest_preview.svg')
    print(f"Cutting DXF written and re-validated: {info['parts']} parts, {info['true_curve_entities']} true arcs/circles, "
          f"difference {info['validation']['symmetric_difference_mm2']:.2f} mm2 -> smartnest_cut.dxf")
    if IN_COLAB:
        for f in ('smartnest_cut.dxf', 'smartnest_material_map.dxf', 'smartnest_job.json', 'smartnest_preview.svg'):
            files.download(f)